In [1]:
# from pathlib import Path
# import sys

# project_root = Path.cwd().parent  # notebooks -> project root
# sys.path.insert(0, str(project_root))

# from clinical_synopsis.embedder import Embedder
# print("embedder import OK")

# to avoid clinical_synopsis.embedder
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "clinical_synopsis"))

from embedder import Embedder
print("embedder import OK")

embedder import OK


In [2]:
import rag as rag

# Look at the first chunk we have in the vector index, take the patient_id attached to that chunk:
pid = rag.vector_documents[0]["patient_id"]

result = rag.rag(
    query="What oncology-related events are documented?",
    patient_id=pid,
    search_type="hybrid",
    num_results=5,
)

len(result["search_results"]), result["search_results"][:2]

(5,
 [{'id': '6e3f2f8c188a50ec8d84a45b499a14a631fcc414',
   'chunk_id': '6e3f2f8c188a50ec8d84a45b499a14a631fcc414',
   'patient_id': '03b93198-d95e-c385-c3a7-80470f411d18',
   'document_id': 'eacfd84f2024e811caae390e056a4e52fefc5249',
   'doc_type': 'oncology_timeline',
   'title': 'Oncology Timeline: Aurora248 Dooley940',
   'heading': 'Oncology Timeline: Aurora248 Dooley940',
   'chunk_text': '- Patient ID: 03b93198-d95e-c385-c3a7-80470f411d18\n- Oncology-related dated events: 52',
   'chunk_index': 0,
   'is_oncology': '1',
   'date_start': '',
   'date_end': '',
   'rrf_score': 0.01639344262295082},
  {'chunk_id': 'f50d5c213d71cb54157be1dbe78d05ff2cb23489',
   'patient_id': '03b93198-d95e-c385-c3a7-80470f411d18',
   'document_id': '304ddea3daed73b3c0c0f47535fef1906ea22987',
   'doc_type': 'oncology_timeline_events',
   'title': 'oncology_timeline_events.csv',
   'heading': 'oncology_timeline_events',
   'chunk_text': "event_type: Observation; date: 2015-10-01T05:29:59-04:00; label:

In [3]:
print("Answer cost (USD):", result["answer_total_cost_usd"])
print("Eval cost (USD):  ", result["eval_total_cost_usd"])
print("Overall cost (USD):", result["overall_total_cost_usd"])

print("Answer tokens (in/out/total):",
      result["prompt_tokens"],
      result["completion_tokens"],
      result["total_tokens"])

print("Eval tokens (in/out/total):",
      result["eval_prompt_tokens"],
      result["eval_completion_tokens"],
      result["eval_total_tokens"])

Answer cost (USD): 0.00315075
Eval cost (USD):   0.0022897499999999997
Overall cost (USD): 0.005440499999999999
Answer tokens (in/out/total): 1951 375 2326
Eval tokens (in/out/total): 2453 100 2553


In [4]:
for i, doc in enumerate(result["search_results"], start=1):
    print("=" * 80)
    print("Rank:", i)
    print("Chunk ID:", doc.get("chunk_id"))
    print("Patient ID:", doc.get("patient_id"))
    print("Doc type:", doc.get("doc_type"))
    print("Title:", doc.get("title"))
    print("Heading:", doc.get("heading"))
    print("Text:", doc.get("chunk_text", "")[:500])

Rank: 1
Chunk ID: 6e3f2f8c188a50ec8d84a45b499a14a631fcc414
Patient ID: 03b93198-d95e-c385-c3a7-80470f411d18
Doc type: oncology_timeline
Title: Oncology Timeline: Aurora248 Dooley940
Heading: Oncology Timeline: Aurora248 Dooley940
Text: - Patient ID: 03b93198-d95e-c385-c3a7-80470f411d18
- Oncology-related dated events: 52
Rank: 2
Chunk ID: f50d5c213d71cb54157be1dbe78d05ff2cb23489
Patient ID: 03b93198-d95e-c385-c3a7-80470f411d18
Doc type: oncology_timeline_events
Title: oncology_timeline_events.csv
Heading: oncology_timeline_events
Text: event_type: Observation; date: 2015-10-01T05:29:59-04:00; label: Cancer Disease Progression; status: Patient's condition improved; resource_id: 3ee3c67c-a4a3-37e1-e49b-f20f32cc14db; source_file: data/prototype/sample50/Aurora248_Dooley940_03b93198-d95e-c385-c3a7-80470f411d18.json
Rank: 3
Chunk ID: 561c5809a7e9251705306dee52b08e9fd64721ce
Patient ID: 03b93198-d95e-c385-c3a7-80470f411d18
Doc type: oncology_timeline
Title: Oncology Timeline: Aurora248 Doole

In [5]:
available_patient_ids = sorted({doc["patient_id"] for doc in rag.vector_documents})
len(available_patient_ids), available_patient_ids[:10]

(50,
 ['03b93198-d95e-c385-c3a7-80470f411d18',
  '0c0f2095-e8ab-7ac4-6ef4-625748255480',
  '0f5704ee-b38b-5a68-449d-9c44806517d0',
  '188e1f01-15b7-d51b-c76d-bdd7772a10e9',
  '25197dc8-9425-1999-5914-f2171b0d4e32',
  '263375ec-5856-81b8-9e51-1cb8e8bcba30',
  '29f6beee-162f-0113-7884-72245814693f',
  '397b2de6-ccd8-858f-bf4a-b6fc379589bd',
  '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678',
  '3af995f1-02a5-07ee-5a7e-e2470a017f1e'])

# 9 patients for ground truth

We pick 9 patients with 3 from each complexity bucket (low, medium, high) for the ground-truth set, in order to cover simple, medium, and complex EHRs.

`n_resources` is the total number of FHIR resources in that patient’s bundle — i.e., how many individual clinical records (Patient, Encounter, Observation, Condition, Procedure, MedicationRequest, DiagnosticReport, etc.) are contained in the JSON file for that patient.

So we see that the higher `n_resources`, the higher `complexity_score`

Remember:
### Note on COMPLEXITY SCORES for each patient

A higher complexity score should reflect more encounters, conditions, procedures, meds, reports for a given patient, as well as a longer follow-up period (e.g., Febrile neutropenia condition gives a clear date (onsetDateTime, recordedDate) and is tied to an encounter, which contributes to follow-up and complexity).

In `sample_mcode_patients.py` a complexity score is computed as:
```python
complexity_score = (
    counts["Encounter"] * 3
    + counts["Observation"] * 1
    + counts["Condition"] * 2
    + counts["Procedure"] * 2
    + counts["MedicationRequest"] * 2
    + counts["MedicationAdministration"] * 2
    + counts["DiagnosticReport"] * 2
    + min(followup_days // 180, 20)
)
```
Which means that:
- Each resource type contributes with a **weight**:
  - Encounters: \(3 \times\) number of encounters (heavier weight).
  - Observations: \(1 \times\) number of observations.
  - Conditions, Procedures, MedicationRequest, MedicationAdministration, DiagnosticReport: each \(2 \times\) their counts.
- Plus a **time component**:
  - `followup_days` is the difference between the first and last clinical dates found in the bundle.
  - `followup_days // 180` converts follow-up into “half-year blocks”.
  - This term is capped at 20, so very long records don’t dominate.



In [6]:
import pandas as pd

manifest_path = project_root / "data" / "processed" / "mcode_breast_sample_50_manifest.csv"

# Load manifest
df = pd.read_csv(manifest_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (50, 17)

Columns:
['filename', 'patient_id', 'patient_name', 'n_resources', 'n_encounters', 'n_observations', 'n_conditions', 'n_procedures', 'n_medication_requests', 'n_medication_administrations', 'n_diagnostic_reports', 'first_date', 'last_date', 'followup_days', 'complexity_score', 'complexity_bucket', 'sample_seed']


In [7]:
df

,filename,patient_id,patient_name,n_resources,n_encounters,n_observations,n_conditions,n_procedures,n_medication_requests,n_medication_administrations,n_diagnostic_reports,first_date,last_date,followup_days,complexity_score,complexity_bucket,sample_seed
0,data/raw/longitudinalMCODEBreast/Corrie32_Boyl...,d65197b3-056a-2136-b584-77f43c29da3f,Corrie32 Boyle917,230,27,80,2,27,20,0,28,2020-12-18T03:22:18-05:00,2022-05-20T18:36:55-04:00,518,317,low,42
1,data/raw/longitudinalMCODEBreast/Florine959_St...,f3739580-797d-ae04-eebf-aeddb2fc2f64,Florine959 Stark857,261,25,119,5,23,11,0,26,2019-06-11T21:25:37-04:00,2022-06-21T05:03:45-04:00,1105,330,low,42
2,data/raw/longitudinalMCODEBreast/Deana43_Baumb...,3f130449-d7db-f118-5bd0-cce81084e911,Deana43 Baumbach677,431,42,206,10,23,18,0,44,2012-12-18T03:09:54-05:00,2022-04-12T04:24:54-04:00,3402,540,low,42
3,data/raw/longitudinalMCODEBreast/Joni720_Stied...,43c173b0-172c-f414-5c62-1bdf4bb33954,Joni720 Stiedemann542,459,47,218,17,29,5,0,51,2014-04-22T07:35:47-04:00,2022-05-17T21:08:31-04:00,2947,579,low,42
4,data/raw/longitudinalMCODEBreast/Santos184_Jas...,64ae3769-65e4-222e-6793-1a3bc14ec682,Santos184 Jaskolski867,468,45,241,4,31,11,0,48,2010-08-02T09:36:48-04:00,2022-05-17T22:36:39-04:00,4306,584,low,42
5,data/raw/longitudinalMCODEBreast/Mónica985_Se...,4736727e-63f4-071a-1516-a49310f5a052,Mónica985 Serrato62,436,61,151,6,54,8,0,62,2018-01-07T15:16:05-05:00,2022-04-21T02:52:37-04:00,1564,602,low,42
6,data/raw/longitudinalMCODEBreast/Maryellen651_...,af3bd539-de27-28d9-9016-f1643d4615c0,Maryellen651 Zboncak558,514,54,238,11,35,18,0,56,2011-08-29T07:05:19-04:00,2022-04-27T18:02:46-04:00,3894,660,low,42
7,data/raw/longitudinalMCODEBreast/Darcie474_Fra...,568ec0af-94fa-521b-012e-88f61f78028f,Darcie474 Frami345,496,69,180,2,59,11,0,71,2015-10-27T05:01:54-04:00,2022-04-25T17:02:50-04:00,2372,686,low,42
8,data/raw/longitudinalMCODEBreast/Jani266_Thiel...,3e693a9a-de55-de2e-aab9-500036bcf04b,Jani266 Thiel172,629,76,258,8,67,14,0,80,2011-05-02T16:40:40-04:00,2022-06-03T06:55:38-04:00,4049,844,low,42
9,data/raw/longitudinalMCODEBreast/Dolores502_Ca...,6fb374e8-33aa-a5ea-f050-b61394dfcb99,Dolores502 Caldera106,872,101,305,13,165,18,1,111,2006-05-18T17:21:05-04:00,2022-06-30T17:36:05-04:00,5887,1244,low,42


In [8]:
# Check bucket distribution
print("Bucket counts in full manifest:")
print(df["complexity_bucket"].value_counts())

# Sample 3 patients from each bucket (low, medium, high)
bucket_targets = {"low": 3, "medium": 3, "high": 3}
selected_rows = []

for bucket, n in bucket_targets.items():
    bucket_df = df[df["complexity_bucket"] == bucket].copy()
    if len(bucket_df) < n:
        raise ValueError(f"Not enough patients in bucket '{bucket}' to sample {n}.")

    # Random sample with a fixed seed for reproducibility
    sampled_bucket = bucket_df.sample(n=n, random_state=42)
    selected_rows.append(sampled_bucket)

selected_df = pd.concat(selected_rows).reset_index(drop=True)

print("\nSelected 9 patients (3 per bucket):")
display(selected_df[["patient_id", "patient_name", "complexity_bucket",
                     "n_resources", "complexity_score"]])

# Just the list of patient_ids for later use
selected_patient_ids = selected_df["patient_id"].tolist()
print("\nSelected patient_ids:", selected_patient_ids)

Bucket counts in full manifest:
complexity_bucket
low       17
high      17
medium    16
Name: count, dtype: int64

Selected 9 patients (3 per bucket):


,patient_id,patient_name,complexity_bucket,n_resources,complexity_score
0,d65197b3-056a-2136-b584-77f43c29da3f,Corrie32 Boyle917,low,230,317
1,f3739580-797d-ae04-eebf-aeddb2fc2f64,Florine959 Stark857,low,261,330
2,4736727e-63f4-071a-1516-a49310f5a052,Mónica985 Serrato62,low,436,602
3,29f6beee-162f-0113-7884-72245814693f,Eula461 Crooks415,medium,1854,2786
4,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,Beth967 Cremin516,medium,1843,2800
5,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,Deeann517 Torp761,medium,2191,3340
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,Francina926 Von197,high,2945,4458
7,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,Rosetta750 Stroman228,high,3055,4536
8,f203e11d-5573-1624-69b8-af8436987b3e,Shawana711 Lakin515,high,3365,4812



Selected patient_ids: ['d65197b3-056a-2136-b584-77f43c29da3f', 'f3739580-797d-ae04-eebf-aeddb2fc2f64', '4736727e-63f4-071a-1516-a49310f5a052', '29f6beee-162f-0113-7884-72245814693f', '41681ed6-efc5-94c0-1bc0-f60b34dbd31b', 'aee216e6-cbe8-eaf2-3241-4bd1e8a01494', 'ecc4a7d0-8838-36b4-44ba-676d5a1f7927', '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678', 'f203e11d-5573-1624-69b8-af8436987b3e']


In [9]:
selected_patient_ids

['d65197b3-056a-2136-b584-77f43c29da3f',
 'f3739580-797d-ae04-eebf-aeddb2fc2f64',
 '4736727e-63f4-071a-1516-a49310f5a052',
 '29f6beee-162f-0113-7884-72245814693f',
 '41681ed6-efc5-94c0-1bc0-f60b34dbd31b',
 'aee216e6-cbe8-eaf2-3241-4bd1e8a01494',
 'ecc4a7d0-8838-36b4-44ba-676d5a1f7927',
 '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678',
 'f203e11d-5573-1624-69b8-af8436987b3e']

In [ ]:
# looks at doc_type counts for selected patients, including oncology-flagged chunks

import sqlite3
import pandas as pd
from pathlib import Path
from IPython.display import display

db_path = Path("../data/retrieval/metadata.db")
conn = sqlite3.connect(db_path)

# Build a parameterized IN clause for SQLite
placeholders = ",".join(["?"] * len(selected_patient_ids))

doc_type_counts_selected = pd.read_sql_query(
    f"""
    SELECT
        chunks.patient_id,
        documents.doc_type,
        COUNT(*) AS n_chunks,
        SUM(CASE WHEN chunks.is_oncology = 1 THEN 1 ELSE 0 END) AS n_oncology_chunks
    FROM chunks
    JOIN documents ON chunks.document_id = documents.document_id
    WHERE chunks.patient_id IN ({placeholders})
    GROUP BY chunks.patient_id, documents.doc_type
    ORDER BY chunks.patient_id, n_chunks DESC
    """,
    conn,
    params=selected_patient_ids,
)

conn.close()

print("Long format: one row per selected patient/doc_type")
display(doc_type_counts_selected)

# Wide pivot: rows = selected patients, columns = doc_types, values = total chunk counts
doc_type_pivot_selected = (
    doc_type_counts_selected
    .pivot(index="patient_id", columns="doc_type", values="n_chunks")
    .fillna(0)
    .astype(int)
)

print("\nChunk counts by doc_type for each selected patient")
display(doc_type_pivot_selected)

# Oncology-only pivot
oncology_pivot_selected = (
    doc_type_counts_selected
    .pivot(index="patient_id", columns="doc_type", values="n_oncology_chunks")
    .fillna(0)
    .astype(int)
)

print("\nOncology-flagged chunk counts by doc_type for each selected patient")
display(oncology_pivot_selected)

# List of doc_types present per selected patient
doc_types_per_selected_patient = (
    doc_type_counts_selected[doc_type_counts_selected["n_chunks"] > 0]
    .groupby("patient_id")["doc_type"]
    .apply(list)
    .reset_index(name="doc_types")
)

print("\nDoc types present for each selected patient")
display(doc_types_per_selected_patient)

Long format: one row per selected patient/doc_type


,patient_id,doc_type,n_chunks,n_oncology_chunks
0,29f6beee-162f-0113-7884-72245814693f,observations,571,14
1,29f6beee-162f-0113-7884-72245814693f,procedures,408,37
2,29f6beee-162f-0113-7884-72245814693f,diagnostic_reports,284,0
3,29f6beee-162f-0113-7884-72245814693f,encounters,231,0
4,29f6beee-162f-0113-7884-72245814693f,oncology_timeline_events,53,53
...,...,...,...,...
76,f3739580-797d-ae04-eebf-aeddb2fc2f64,oncology_timeline_events,22,22
77,f3739580-797d-ae04-eebf-aeddb2fc2f64,medications,11,0
78,f3739580-797d-ae04-eebf-aeddb2fc2f64,patient_overview,9,9
79,f3739580-797d-ae04-eebf-aeddb2fc2f64,oncology_timeline,5,5



Chunk counts by doc_type for each selected patient


doc_type,conditions,diagnostic_reports,encounters,medications,observations,oncology_timeline,oncology_timeline_events,patient_overview,procedures
patient_id,,,,,,,,,
29f6beee-162f-0113-7884-72245814693f,42,284,231,17,571,9,53,9,408
3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,95,417,273,119,1133,6,26,9,651
41681ed6-efc5-94c0-1bc0-f60b34dbd31b,56,250,197,41,565,9,49,9,465
4736727e-63f4-071a-1516-a49310f5a052,6,62,61,8,151,9,47,9,54
aee216e6-cbe8-eaf2-3241-4bd1e8a01494,50,336,287,45,597,9,52,9,500
d65197b3-056a-2136-b584-77f43c29da3f,2,28,27,20,80,6,29,9,27
ecc4a7d0-8838-36b4-44ba-676d5a1f7927,115,394,262,72,970,9,48,9,760
f203e11d-5573-1624-69b8-af8436987b3e,95,404,270,40,1322,9,52,9,791
f3739580-797d-ae04-eebf-aeddb2fc2f64,5,26,25,11,119,5,22,9,23



Oncology-flagged chunk counts by doc_type for each selected patient


doc_type,conditions,diagnostic_reports,encounters,medications,observations,oncology_timeline,oncology_timeline_events,patient_overview,procedures
patient_id,,,,,,,,,
29f6beee-162f-0113-7884-72245814693f,2,0,0,0,14,9,53,9,37
3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,2,0,0,0,14,6,26,0,10
41681ed6-efc5-94c0-1bc0-f60b34dbd31b,2,0,0,0,11,9,49,9,36
4736727e-63f4-071a-1516-a49310f5a052,1,0,0,0,11,9,47,9,35
aee216e6-cbe8-eaf2-3241-4bd1e8a01494,2,0,0,0,14,9,52,9,36
d65197b3-056a-2136-b584-77f43c29da3f,1,0,0,0,11,6,29,9,17
ecc4a7d0-8838-36b4-44ba-676d5a1f7927,2,0,0,0,10,9,48,9,36
f203e11d-5573-1624-69b8-af8436987b3e,2,0,0,0,14,9,52,0,36
f3739580-797d-ae04-eebf-aeddb2fc2f64,2,0,0,0,10,5,22,9,10



Doc types present for each selected patient


,patient_id,doc_types
0,29f6beee-162f-0113-7884-72245814693f,"[observations, procedures, diagnostic_reports,..."
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,"[observations, procedures, diagnostic_reports,..."
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,"[observations, procedures, diagnostic_reports,..."
3,4736727e-63f4-071a-1516-a49310f5a052,"[observations, diagnostic_reports, encounters,..."
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,"[observations, procedures, diagnostic_reports,..."
5,d65197b3-056a-2136-b584-77f43c29da3f,"[observations, oncology_timeline_events, diagn..."
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,"[observations, procedures, diagnostic_reports,..."
7,f203e11d-5573-1624-69b8-af8436987b3e,"[observations, procedures, diagnostic_reports,..."
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,"[observations, diagnostic_reports, encounters,..."


In [ ]:
# check that all selected patients have the same set of doc_types (ignoring order)

import pandas as pd
from IPython.display import display

# Example: doc_types_per_selected_patient already computed, like:
# doc_types_per_selected_patient = (
#     doc_type_counts_selected[doc_type_counts_selected["n_chunks"] > 0]
#     .groupby("patient_id")["doc_type"]
#     .apply(list)
#     .reset_index(name="doc_types")
# )

# print("Doc types per selected patient:")
# display(doc_types_per_selected_patient)

# Convert each list of doc_types into a frozenset so order doesn't matter
doc_types_per_selected_patient["doc_types_set"] = (
    doc_types_per_selected_patient["doc_types"]
    .apply(lambda lst: frozenset(lst))
)

# Use the first patient's set as reference
reference_set = doc_types_per_selected_patient["doc_types_set"].iloc[0]
print("\nReference doc_types_set:", reference_set)

# Check equality against the reference
doc_types_per_selected_patient["matches_reference"] = (
    doc_types_per_selected_patient["doc_types_set"] == reference_set
)

print("\nEquality check vs reference:")
display(doc_types_per_selected_patient[["patient_id", "doc_types", "matches_reference"]])

# Summarize: how many distinct doc_types sets exist?
unique_sets = doc_types_per_selected_patient["doc_types_set"].unique()
print("\nNumber of distinct doc_types sets among selected patients:", len(unique_sets))

if len(unique_sets) > 1:
    print("Patients grouped by their doc_types_set:")
    # For readability, show each unique set and which patients have it
    for s in unique_sets:
        patients_with_set = doc_types_per_selected_patient[
            doc_types_per_selected_patient["doc_types_set"] == s
        ]["patient_id"].tolist()
        print(f"\nSet: {sorted(list(s))}")
        print("Patients:", patients_with_set)


Reference doc_types_set: frozenset({'oncology_timeline', 'procedures', 'observations', 'diagnostic_reports', 'encounters', 'oncology_timeline_events', 'medications', 'conditions', 'patient_overview'})

Equality check vs reference:


,patient_id,doc_types,matches_reference
0,29f6beee-162f-0113-7884-72245814693f,"[observations, procedures, diagnostic_reports,...",True
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,"[observations, procedures, diagnostic_reports,...",True
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,"[observations, procedures, diagnostic_reports,...",True
3,4736727e-63f4-071a-1516-a49310f5a052,"[observations, diagnostic_reports, encounters,...",True
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,"[observations, procedures, diagnostic_reports,...",True
5,d65197b3-056a-2136-b584-77f43c29da3f,"[observations, oncology_timeline_events, diagn...",True
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,"[observations, procedures, diagnostic_reports,...",True
7,f203e11d-5573-1624-69b8-af8436987b3e,"[observations, procedures, diagnostic_reports,...",True
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,"[observations, diagnostic_reports, encounters,...",True



Number of distinct doc_types sets among selected patients: 1


In [12]:
# look at all oncology chunks in chunks_df:

import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("../data/retrieval/metadata.db")
patient_ids = selected_patient_ids

conn = sqlite3.connect(db_path)

# Multiple patient_ids: build an IN (...) placeholder list.
if not patient_ids:
    chunks_df = pd.DataFrame()  # avoid invalid SQL: IN ()
else:
    placeholders = ",".join(["?"] * len(patient_ids))
    query = f"""
    SELECT
        chunks.patient_id,
        documents.doc_type,
        documents.title,
        chunks.heading,
        chunks.chunk_id,
        chunks.is_oncology,
        chunks.chunk_text
    FROM chunks
    JOIN documents ON chunks.document_id = documents.document_id
    WHERE chunks.patient_id IN ({placeholders})
    """
    chunks_df = pd.read_sql_query(query, conn, params=patient_ids)

conn.close()

display(chunks_df.head())
print("Number of rows:", len(chunks_df))

onc_chunks = chunks_df[chunks_df["is_oncology"] == 1]
display(onc_chunks.head())
print("Number of oncology chunks:", len(onc_chunks))

,patient_id,doc_type,title,heading,chunk_id,is_oncology,chunk_text
0,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,82fbc5fb0df955ed9b5861c956dd9d5cdd0a1aa8,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
1,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,cd9bf67f542dee2c5c6eb4e889086745d239ff7b,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
2,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,c6f4af530450ec4d38f7f478693b4d62c2b3467c,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
3,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,9d5bdbca525ce28a75b1ad7deff73853cc1b20eb,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
4,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,250fb82c086a015ec8fe85d5feeb46806e632e86,0,patient_id: 29f6beee-162f-0113-7884-7224581469...


Number of rows: 14390


,patient_id,doc_type,title,heading,chunk_id,is_oncology,chunk_text
0,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,82fbc5fb0df955ed9b5861c956dd9d5cdd0a1aa8,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
21,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,34e083fbe248b34ab7ff75e989fc91da40a6b511,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
931,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,6c44d78bfe33c70133759592f49f88f9cd26734c,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
932,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,4f181909c2fad14ebbdc20f28f1af14d5dfb89f9,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
933,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,b02dbcb96bafeb63dc73b3c40a9f184cb0e781bf,1,patient_id: 29f6beee-162f-0113-7884-7224581469...


Number of oncology chunks: 890


In [13]:
onc_chunks["doc_type"].value_counts()

doc_type
oncology_timeline_events    378
procedures                  253
observations                109
oncology_timeline            71
patient_overview             63
conditions                   16
Name: count, dtype: int64

In [14]:
onc_chunks.groupby("patient_id")["doc_type"].count()

patient_id
29f6beee-162f-0113-7884-72245814693f    124
3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678     58
41681ed6-efc5-94c0-1bc0-f60b34dbd31b    116
4736727e-63f4-071a-1516-a49310f5a052    112
aee216e6-cbe8-eaf2-3241-4bd1e8a01494    122
d65197b3-056a-2136-b584-77f43c29da3f     73
ecc4a7d0-8838-36b4-44ba-676d5a1f7927    114
f203e11d-5573-1624-69b8-af8436987b3e    113
f3739580-797d-ae04-eebf-aeddb2fc2f64     58
Name: doc_type, dtype: int64

In [15]:
onc_chunks.groupby("doc_type")["patient_id"].value_counts()

doc_type                  patient_id                          
conditions                29f6beee-162f-0113-7884-72245814693f     2
                          ecc4a7d0-8838-36b4-44ba-676d5a1f7927     2
                          41681ed6-efc5-94c0-1bc0-f60b34dbd31b     2
                          f203e11d-5573-1624-69b8-af8436987b3e     2
                          aee216e6-cbe8-eaf2-3241-4bd1e8a01494     2
                          f3739580-797d-ae04-eebf-aeddb2fc2f64     2
                          3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678     2
                          d65197b3-056a-2136-b584-77f43c29da3f     1
                          4736727e-63f4-071a-1516-a49310f5a052     1
observations              3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678    14
                          f203e11d-5573-1624-69b8-af8436987b3e    14
                          aee216e6-cbe8-eaf2-3241-4bd1e8a01494    14
                          29f6beee-162f-0113-7884-72245814693f    14
                          41681ed6-efc5-

In [16]:
chunks_df.columns

Index(['patient_id', 'doc_type', 'title', 'heading', 'chunk_id', 'is_oncology',
       'chunk_text'],
      dtype='str')

# Shared set of question

That’s a very good sign: it means all 9 selected patients have the **same set of document types** (even if the order in the lists differs).

Practically, this gives you a clean foundation for your evaluation set:

- Every selected patient has all of:
  - `patient_overview`
  - `conditions`
  - `medications`
  - `procedures`
  - `observations`
  - `diagnostic_reports`
  - `encounters`
  - `oncology_timeline_events`
  - `oncology_timeline` (plus whatever else is in that set).

So you can safely design a **shared set of question templates** and apply them across all 9 patients, only skipping oncology-specific ones for those who truly have no oncology-flagged content. [youtube](https://www.youtube.com/watch?v=IYx4O42dHd0)

A natural next step, given this:

- Define 3–4 question types that map onto these doc_types (e.g., overview, conditions, medications, oncology history).
- Then, for each of your 9 patients, label gold_chunk_ids for those questions using the doc_type + is_oncology filters you already have.

If you want, I can propose a concrete list of 4 question templates that directly correspond to these doc_types and are suitable for all 9 patients.

Great — here’s a concrete 4-question core set plus a 5-question extended set, both designed to work across all 9 selected patients, along with how to choose `gold_chunk_ids` for each. Because all patients share the same doc_types, you can apply this uniformly. 

### Core 4-question set (recommended baseline)

Use this for your main evaluation; it balances general and oncology content and stays relatively easy to annotate.

1. **Patient overview**

   - Question:  
     “Give a concise overview of this patient’s medical background and current care context.”
   - Primary doc_types to draw gold chunks from:  
     `patient_overview`, `encounters`, `conditions`, `oncology_timeline_events`. 
   - Gold chunk guidance:
     - Include 2–4 chunks that together cover:
       - Key chronic conditions. 
       - Major past procedures or events if they define the patient’s course. 
       - A high-level oncology context if applicable (e.g., diagnosis, line of therapy). 
     - Prefer chunks that are explicitly summary-like (e.g., overview notes) over raw measurements.

2. **Conditions**

   - Question:  
     “What are the patient’s main diagnosed conditions?”
   - Primary doc_types:  
     `conditions`, possibly `encounters` or `patient_overview` if they contain canonical lists. 
   - Gold chunk guidance:
     - Select chunks that list named diagnoses (problem lists, diagnosis sections, structured condition records). 
     - If conditions evolve (e.g., disease progression or resolved conditions), include chunks that clearly mark the current status.

3. **Medications**

   - Question:  
     “What medications is the patient taking or has recently taken?”
   - Primary doc_types:  
     `medications`, optionally `encounters` or `oncology_timeline_events` if they record regimens. 
   - Gold chunk guidance:
     - Focus on chunks that list active or recent meds (lists, med history sections). 
     - If oncology regimens are recorded in timeline events rather than meds, include those chunks too. 

4. **Oncology timeline**

   - Question:  
     “Summarize the patient’s oncology-related timeline, including major events and treatments.”
   - Primary doc_types:  
     `oncology_timeline_events`, `oncology_timeline`, plus supporting `encounters` or `diagnostic_reports` if needed. 
   - Gold chunk guidance:
     - Choose 3–6 chunks covering:
       - Initial diagnosis/first cancer-related event.
       - Key treatment starts/changes (lines of therapy, major procedures). 
       - Notable response/progression events or critical findings (e.g., scan results summaries). 
     - Try to keep the subset coherent and roughly chronological, so a model could reconstruct a timeline from them.

### Extended 5-question set (adds a synthesis question)

For a richer benchmark, add this fifth question; it forces multi-doc-type reasoning.

5. **Cross-document clinical priorities**

   - Question:  
     “What clinical issues appear to be most important for this patient right now?”
   - Primary doc_types:  
     `conditions`, `medications`, `encounters`, `observations`, `diagnostic_reports`, and `oncology_timeline_events`. 
   - Gold chunk guidance:
     - Pick 4–8 chunks such that:
       - Each chunk contributes a distinct piece of evidence (e.g., a condition, a treatment, a key observation/report). 
       - Together they justify why certain issues are “most important” (e.g., active malignancy on treatment, uncontrolled comorbidity, acute complication). 
     - Explicitly favor chunks that are:
       - Recent in time.
       - Clearly interpretable (e.g., summary sections rather than isolated lab values), unless a lab/report is itself critical.

### How to pick gold_chunk_ids in practice

For each patient and each question:

- Step 1: Filter by doc_type and (if you have it) `is_oncology` flag or recency indicators. 
- Step 2: From the filtered subset, manually inspect and choose only the chunks that:
  - Directly answer the question.
  - Are reasonably self-contained (the model doesn’t need many unrelated chunks to interpret them).
- Step 3: Store:
  - `question_text`
  - `patient_id`
  - `gold_chunk_ids` (list of chunk IDs)
  - Optionally a short `rationale` free-text explaining why these chunks were selected (can be handy for later analysis).

This gives you a consistent scheme across all 9 patients: every patient gets the same 4 core questions, and optionally the 5th synthesis question, with gold_chunk_ids drawn from matching doc_types. 

Do you already have a column that marks “recent” vs “historical” chunks (e.g., encounter date), or should I suggest a simple heuristic for recency using only what’s in your `chunks` table?

#### 1. **Patient overview**

This filters candidate chunks for one patient for the Patient Overview question, prioritizing summary-like doc types and creating a readable preview. Filtering with isin(...) and sorting with sort_values(...) are standard Pandas patterns for this kind of review table:

In [17]:
import pandas as pd
from IPython.display import display

QUESTION_TYPE = "patient_overview"
QUESTION_TEXT = "Give a concise overview of this patient’s medical background and current care context."

overview_headings = [
    "Recent Condition",
    "Recent Results",
    "Procedures",
]

# Summary-like chunks from patient_overview
mask_po_headings = (
    (chunks_df["doc_type"] == "patient_overview")
    & (chunks_df["heading"].isin(overview_headings))
)

# Optional: include conditions.csv as secondary context
mask_conditions = chunks_df["doc_type"] == "conditions"

relevant_mask = mask_po_headings #| mask_conditions  # or just mask_po_headings if you want it narrower

patient_overview_gold = (
    chunks_df.loc[relevant_mask, ["patient_id", "chunk_id"]]
    .groupby("patient_id")["chunk_id"]
    .apply(list)
    .reset_index()
    .rename(columns={"chunk_id": "gold_chunk_ids"})
)

patient_overview_gold["question_type"] = QUESTION_TYPE
patient_overview_gold["question_text"] = QUESTION_TEXT

patient_overview_gold = patient_overview_gold[
    ["patient_id", "question_type", "question_text", "gold_chunk_ids"]
]

display(patient_overview_gold)

,patient_id,question_type,question_text,gold_chunk_ids
0,29f6beee-162f-0113-7884-72245814693f,patient_overview,Give a concise overview of this patient’s medi...,"[abc678e3f0fdb2313c03b92ff62bf06f08120f7b, c1b..."
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,patient_overview,Give a concise overview of this patient’s medi...,"[a970cc1ae6abca57a8813b4cc2aea2df1d41cde1, 519..."
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,patient_overview,Give a concise overview of this patient’s medi...,"[0590211551c10b6259f5a9062003667e93e8f4e6, 9c7..."
3,4736727e-63f4-071a-1516-a49310f5a052,patient_overview,Give a concise overview of this patient’s medi...,"[63f01e07f0f326ce090ed1d0e1e9cb9ddb88d53b, 49c..."
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,patient_overview,Give a concise overview of this patient’s medi...,"[aff56cb2f4bb43dff398ad5e744d1f5a364b7670, 280..."
5,d65197b3-056a-2136-b584-77f43c29da3f,patient_overview,Give a concise overview of this patient’s medi...,"[abc3ffd40cbb13d210ac910a0f5aaa43b998a587, e84..."
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,patient_overview,Give a concise overview of this patient’s medi...,"[aee14674ce11ab5daa05de7c09f81db1f25365a4, 95b..."
7,f203e11d-5573-1624-69b8-af8436987b3e,patient_overview,Give a concise overview of this patient’s medi...,"[1f9771cf6cdb8353d65819a133ea34b11b3b8e58, 78c..."
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,patient_overview,Give a concise overview of this patient’s medi...,"[a9165d0158523ea3b0c882ba42cd1f37f02f067d, c59..."


#### 2. Conditions

In [68]:
QUESTION_TYPE = "conditions"
QUESTION_TEXT = "What are the patient’s main diagnosed conditions?"

condition_headings = ["Recent Conditions", "Recent Results"]

mask_po_conditions = (                              # only the patient_overview chunks with conditions
    (chunks_df["doc_type"] == "patient_overview")
    & (chunks_df["heading"].isin(condition_headings))
)

# For now, focus gold on patient_overview only
relevant_mask = mask_po_conditions

conditions_gold = (
    chunks_df.loc[relevant_mask, ["patient_id", "chunk_id"]]
    .groupby("patient_id")["chunk_id"]
    .apply(list)
    .reset_index()
    .rename(columns={"chunk_id": "gold_chunk_ids"})
)

conditions_gold["question_type"] = QUESTION_TYPE
conditions_gold["question_text"] = QUESTION_TEXT

conditions_gold = conditions_gold[
    ["patient_id", "question_type", "question_text", "gold_chunk_ids"]
]

display(conditions_gold)

,patient_id,question_type,question_text,gold_chunk_ids
0,29f6beee-162f-0113-7884-72245814693f,conditions,What are the patient’s main diagnosed conditions?,"[14326c002878a932e87427ea2a9ba2b6e6ad1404, abc..."
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,conditions,What are the patient’s main diagnosed conditions?,"[772ce44ecf1664ce3cd0f321962e31b1afeaf658, a97..."
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,conditions,What are the patient’s main diagnosed conditions?,"[8123905ce9ef5c2a100a0400f86029469183dbb2, 059..."
3,4736727e-63f4-071a-1516-a49310f5a052,conditions,What are the patient’s main diagnosed conditions?,"[7ac2bda6270220cb79ace629a38ef8b5a13ddf31, 63f..."
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,conditions,What are the patient’s main diagnosed conditions?,"[e733276d7db1b542f556b25839e3602be1339cbc, aff..."
5,d65197b3-056a-2136-b584-77f43c29da3f,conditions,What are the patient’s main diagnosed conditions?,"[45b0f62c2348b66d770620e4e5acfe630cf41f96, abc..."
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,conditions,What are the patient’s main diagnosed conditions?,"[05d547f22443d60a0d31fa1776bed869da7459cd, aee..."
7,f203e11d-5573-1624-69b8-af8436987b3e,conditions,What are the patient’s main diagnosed conditions?,"[f73c3623d5df217075882e25106a9d6388dde40e, 1f9..."
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,conditions,What are the patient’s main diagnosed conditions?,"[4c17e4784d257f85e9fd1ddd74054e5dc3adae9f, a91..."


#### 3. Medications

We can have different masks for Conditions and Medications, 
```
# For now, focus gold on patient_overview only
relevant_mask = mask_po_conditions
```
and
```
# Gold includes both, but we know patient_overview is primary
relevant_mask = mask_doc_type | mask_heading
```
because:
- For Conditions, you’ve explicitly said:
    - The clinically meaningful notion of “main diagnosed conditions” for the eval question is the curated list in patient_overview (Recent Conditions/Results).
    - The full conditions.csv contains many extra entries (social findings, old minor conditions) that you don’t want the model to treat as “main conditions”.
- For Medications, your description was:
    - The summary in patient_overview is what you want to show to the user.
    - But the structured medications.csv rows are also directly relevant to “medications the patient is taking or has recently taken” (they are literally the meds list, not social findings).

But to keep it really simple and symmetric, do summary-only for both Conditions and Medications, and treat CSVs strictly as secondary detail in both eval and app 

In [19]:
QUESTION_TYPE = "medications"
QUESTION_TEXT = "What medications is the patient taking or has recently taken?"

med_doc_types = ["medications"]
med_headings = ["Medications"]   # adjust to exact heading text in patient_overview.md

mask_po_medications = (                              # only the patient_overview chunks with medications
    (chunks_df["doc_type"] == "patient_overview")
    & (chunks_df["heading"].isin(med_headings))
)

# For now, focus gold on patient_overview only
relevant_mask = mask_po_medications

medications_gold = (
    chunks_df.loc[relevant_mask, ["patient_id", "chunk_id"]]
    .groupby("patient_id")["chunk_id"]
    .apply(list)
    .reset_index()
    .rename(columns={"chunk_id": "gold_chunk_ids"})
)

medications_gold["question_type"] = QUESTION_TYPE
medications_gold["question_text"] = QUESTION_TEXT

medications_gold = medications_gold[
    ["patient_id", "question_type", "question_text", "gold_chunk_ids"]
]

display(medications_gold)

,patient_id,question_type,question_text,gold_chunk_ids
0,29f6beee-162f-0113-7884-72245814693f,medications,What medications is the patient taking or has ...,[313381cf2c6383b0d66557ee95469ebb92cf7cc1]
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,medications,What medications is the patient taking or has ...,[3f1c94dc52eb9d6748171ca99d8d35ee5b0cc2cd]
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,medications,What medications is the patient taking or has ...,[392065e296a5586b78eb9e2a43d08df5220cb956]
3,4736727e-63f4-071a-1516-a49310f5a052,medications,What medications is the patient taking or has ...,[7d4fb354f104d3cc11273d15182d3f548ca4031b]
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,medications,What medications is the patient taking or has ...,[3b6a76df9116342cca819342b5a77a629c04d636]
5,d65197b3-056a-2136-b584-77f43c29da3f,medications,What medications is the patient taking or has ...,[24c7158a723f3ec52139411b806ca04fe110024e]
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,medications,What medications is the patient taking or has ...,[e3fbadc0fd8c550e6501a326f6222e5c97538c8d]
7,f203e11d-5573-1624-69b8-af8436987b3e,medications,What medications is the patient taking or has ...,[e0893f6f1f540051fea4cc26cef7068cb59b41c6]
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,medications,What medications is the patient taking or has ...,[be283721d4470a4515d90787d7ce60f5a92db51e]


#### 4. **Oncology timeline**

   - Question:  
     “Summarize the patient’s oncology-related timeline, including major events and treatments.”
   - Primary doc_types:  
     `oncology_timeline_events`, `oncology_timeline`, plus supporting `encounters` or `diagnostic_reports` if needed. 
   - Gold chunk guidance:
     - Choose 3–6 chunks covering:
       - Initial diagnosis/first cancer-related event.
       - Key treatment starts/changes (lines of therapy, major procedures). 
       - Notable response/progression events or critical findings (e.g., scan results summaries). 
     - Try to keep the subset coherent and roughly chronological, so a model could reconstruct a timeline from them.


Given your data:

oncology_timeline_events has many granular entries (30+ per patient).

oncology_timeline has far fewer (e.g., 9 chunks) and is explicitly designed as a patient-level timeline summary.

patient_overview.md has Recent Conditions, which give high-level disease context but not a full chronological narrative.

I would:

Include all chunks with doc_type == "oncology_timeline".
These are your primary gold chunks: they encode the backbone of the cancer journey.

Optionally include Recent Condition chunks from patient_overview only if they add important context that’s missing from oncology_timeline, e.g.:

First mention of a malignancy that doesn’t clearly appear in the timeline summary.

Major comorbid oncologic conditions that help interpret the timeline (e.g., multiple primaries).

In [20]:
QUESTION_TYPE = "oncology_timeline"
QUESTION_TEXT = (
    "Summarize the patient’s oncology-related timeline, including major events and treatments."
)

timeline_doc_types = ["oncology_timeline"]

mask_timeline = chunks_df["doc_type"].isin(timeline_doc_types)

oncology_timeline_gold = (
    chunks_df.loc[mask_timeline, ["patient_id", "chunk_id"]]
    .groupby("patient_id")["chunk_id"]
    .apply(list)
    .reset_index()
    .rename(columns={"chunk_id": "gold_chunk_ids"})
)

oncology_timeline_gold["question_type"] = QUESTION_TYPE
oncology_timeline_gold["question_text"] = QUESTION_TEXT

oncology_timeline_gold = oncology_timeline_gold[
    ["patient_id", "question_type", "question_text", "gold_chunk_ids"]
]

display(oncology_timeline_gold)

,patient_id,question_type,question_text,gold_chunk_ids
0,29f6beee-162f-0113-7884-72245814693f,oncology_timeline,Summarize the patient’s oncology-related timel...,"[c21e06064148b6da08db071c8263d1da295ed93c, 17f..."
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,oncology_timeline,Summarize the patient’s oncology-related timel...,"[22d46b9aab128e2a1acdc0d26d872f8bd10f7f79, 674..."
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,oncology_timeline,Summarize the patient’s oncology-related timel...,"[e5ca0cdb0930f81b8adfb7fa94b2b6cb38539862, 441..."
3,4736727e-63f4-071a-1516-a49310f5a052,oncology_timeline,Summarize the patient’s oncology-related timel...,"[5548c8e86138b4a33e1d8a12c518e43dcd667f50, 441..."
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,oncology_timeline,Summarize the patient’s oncology-related timel...,"[30263ea4c834c2be4745a107b2f8386cbd98a1e4, 2fe..."
5,d65197b3-056a-2136-b584-77f43c29da3f,oncology_timeline,Summarize the patient’s oncology-related timel...,"[0ab557068bb85b16a4aad4d40f2a6c83cd45d5b3, 1fb..."
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,oncology_timeline,Summarize the patient’s oncology-related timel...,"[4105225d21ffe94cb84279be1b62e59149f4ed35, fde..."
7,f203e11d-5573-1624-69b8-af8436987b3e,oncology_timeline,Summarize the patient’s oncology-related timel...,"[c9c3776fbdbf55e34536d8b71d32ba80b80f0ad2, 98d..."
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,oncology_timeline,Summarize the patient’s oncology-related timel...,"[d963a6518333e3e87c905fcecd599bf3f90c3f21, 330..."


## Retrieval evaluation for 4 questions


In [69]:
import math
import rag as rag

def get_ranked_chunk_ids(query, patient_id, search_type, k=5):
    """Run the chosen search and return top-k chunk_ids in rank order."""
    if search_type == "lexical":
        results = rag.search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    elif search_type == "semantic":
        results = rag.semantic_search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    elif search_type == "hybrid":
        results = rag.hybrid_search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    else:
        raise ValueError("search_type must be one of: lexical, semantic, hybrid")

    return [doc["chunk_id"] for doc in results]

def hit_rate_at_k(retrieved_ids, relevant_ids, k):
    """Hit@K: 1 if ANY relevant chunk appears in top K, else 0."""
    relevant = set(relevant_ids)
    top_k = retrieved_ids[:k]
    return int(any(doc_id in relevant for doc_id in top_k))

def mrr_at_k(retrieved_ids, relevant_ids, k):
    """MRR@K: 1/rank of first relevant chunk in top K, else 0."""
    relevant = set(relevant_ids)
    for rank, doc_id in enumerate(retrieved_ids[:k], start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

In [70]:
import math

def eval_search_type_df(gold_df, search_type, k=5):
    """
    Evaluate a given search_type over all rows in a gold DataFrame.

    gold_df columns:
      - patient_id
      - question_text  (or 'question' if you prefer)
      - gold_chunk_ids (list of chunk_ids)
    """
    hits = []
    mrrs = []

    for _, row in gold_df.iterrows():
        patient_id = row["patient_id"]
        question = row["question_text"]   # matches your patient_overview_gold schema
        gold_ids = row["gold_chunk_ids"]

        retrieved_ids = get_ranked_chunk_ids(
            query=question,
            patient_id=patient_id,
            search_type=search_type,
            k=k,
        )

        hits.append(hit_rate_at_k(retrieved_ids, gold_ids, k))
        mrrs.append(mrr_at_k(retrieved_ids, gold_ids, k))

    hit_rate = sum(hits) / len(hits) if hits else math.nan
    mrr = sum(mrrs) / len(mrrs) if mrrs else math.nan

    return hit_rate, mrr

In [71]:
# Filter just the patient_overview rows if you have multiple question_types
po_gold = patient_overview_gold[
    patient_overview_gold["question_type"] == "patient_overview"
]

for st in ["lexical", "semantic", "hybrid"]:
    hr, mrr = eval_search_type_df(po_gold, st, k=5)
    print(f"[patient_overview] {st}: Hit@5 = {hr:.3f}, MRR@5 = {mrr:.3f}")

[patient_overview] lexical: Hit@5 = 0.778, MRR@5 = 0.189
[patient_overview] semantic: Hit@5 = 1.000, MRR@5 = 0.289
[patient_overview] hybrid: Hit@5 = 1.000, MRR@5 = 0.313


In [72]:
# Filter just the conditions rows if you have multiple question_types
po_gold = conditions_gold[
    conditions_gold["question_type"] == "conditions"
]

for st in ["lexical", "semantic", "hybrid"]:
    hr, mrr = eval_search_type_df(po_gold, st, k=5)
    print(f"[conditions_gold] {st}: Hit@5 = {hr:.3f}, MRR@5 = {mrr:.3f}")

[conditions_gold] lexical: Hit@5 = 0.111, MRR@5 = 0.037
[conditions_gold] semantic: Hit@5 = 1.000, MRR@5 = 0.422
[conditions_gold] hybrid: Hit@5 = 0.556, MRR@5 = 0.231


In [73]:
# Filter just the medications rows if you have multiple question_types
po_gold = medications_gold[
    medications_gold["question_type"] == "medications"
]

for st in ["lexical", "semantic", "hybrid"]:
    hr, mrr = eval_search_type_df(po_gold, st, k=5)
    print(f"[medications_gold] {st}: Hit@5 = {hr:.3f}, MRR@5 = {mrr:.3f}")

[medications_gold] lexical: Hit@5 = 0.000, MRR@5 = 0.000
[medications_gold] semantic: Hit@5 = 1.000, MRR@5 = 0.741
[medications_gold] hybrid: Hit@5 = 0.333, MRR@5 = 0.081


In [34]:
# Filter just the oncology_timeline rows if you have multiple question_types
po_gold = oncology_timeline_gold[
    oncology_timeline_gold["question_type"] == "oncology_timeline"
]

for st in ["lexical", "semantic", "hybrid"]:
    hr, mrr = eval_search_type_df(po_gold, st, k=5)
    print(f"[oncology_timeline_gold] {st}: Hit@5 = {hr:.3f}, MRR@5 = {mrr:.3f}")

[oncology_timeline_gold] lexical: Hit@5 = 1.000, MRR@5 = 1.000
[oncology_timeline_gold] semantic: Hit@5 = 1.000, MRR@5 = 0.944
[oncology_timeline_gold] hybrid: Hit@5 = 1.000, MRR@5 = 1.000


### Results

These numbers make sense given how you changed the gold sets, and they’re actually telling you something very useful about your retrievers:

- **Patient Overview** behaves like before, because the gold rule change mostly affected which chunks are labeled, not the retrieval difficulty at k=5.
- **Conditions and Medications** now focus gold on the `patient_overview` summary chunks, which lexical retrieval doesn’t find in the top‑5 at all, while semantic retrieval does much better.

Let’s interpret each block.

Patient overview (unchanged pattern)

You now have gold for Patient Overview restricted to the three `patient_overview` headings (Recent Condition, Recent Results, Procedures), but the metrics stayed:

- Lexical: Hit@5 = 0.778, MRR@5 = 0.189
- Semantic: Hit@5 = 1.000, MRR@5 = 0.289
- Hybrid: Hit@5 = 1.000, MRR@5 = 0.313

Interpretation:

- For about 7 of 9 patients, lexical finds at least one gold summary chunk in top‑5; for 2, it misses. And when it hits, the first hit is usually near the bottom of the top‑5. [1706][1707]
- Semantic and hybrid always find a gold chunk and rank it closer to the top (MRR ~ 0.29–0.31), meaning the first relevant chunk is around rank 3–4. [1706][1709]

So the earlier conclusion still holds: **Patient Overview should favor semantic or hybrid retrieval**, not lexical alone.

Conditions (summary-only gold now)

With gold now focused on `patient_overview` summary chunks (mask_po_conditions), metrics are:

- Lexical: Hit@5 = 0.000, MRR@5 = 0.000
- Semantic: Hit@5 = 0.778, MRR@5 = 0.289
- Hybrid: Hit@5 = 0.333, MRR@5 = 0.106

Interpretation:

- **Lexical**:  
  Hit@5 = 0.000 means for none of the 9 Conditions queries does lexical retrieve the patient_overview summary chunk in the top‑5. So it’s not finding the curated Recent Conditions/Results chunks you care about at k=5. [1707][1691]
- **Semantic**:
  Hit@5 = 0.778 → semantic retrieval finds the summary chunk in top‑5 for ~7 of 9 patients.  
  MRR@5 = 0.289 → first hit is around rank 3–4 when it hits.
- **Hybrid**:
  Hit@5 = 0.333 → hybrid only finds the summary chunk in top‑5 for ~3 of 9.  
  MRR@5 = 0.106 → first hit is usually much lower (or missing), so hybrid isn’t helping here.

This is a big change from the earlier “all 1s” scenario, and it reflects your **new gold definition**:

- Before, gold included every conditions.csv chunk → lexical always hit and MRR@5=1.0 simply because any conditions.csv chunk was gold.
- Now, gold targets a specific summary chunk in patient_overview → lexical isn’t retrieving that chunk at all in top‑5; semantic partially does; hybrid mixes signals and sometimes hurts.

So for the **Conditions eval question** (“main diagnosed conditions”):

- Lexical retrieval is no longer adequate at k=5.  
- **Semantic retrieval is clearly better** at finding the right summary chunk. [1691][1707]
- Hybrid may need different tuning (e.g., different RRF k or weighting) to avoid drowning the summary chunk in conditions.csv hits.

Medications (summary-only primary gold)

Metrics:

- Lexical: Hit@5 = 0.000, MRR@5 = 0.000
- Semantic: Hit@5 = 1.000, MRR@5 = 0.741
- Hybrid: Hit@5 = 0.333, MRR@5 = 0.081

Interpretation:

- **Lexical**:  
  Again, Hit@5 = 0.000 → lexical doesn’t retrieve the patient_overview Medications summary chunk in the top‑5, for any patient. It’s likely focused on meds.csv entries and missing the summary chunk entirely. [1707]
- **Semantic**:
  Hit@5 = 1.000 → semantic always retrieves the summary chunk in top‑5 for Medications queries.  
  MRR@5 = 0.741 → average first-hit rank ≈ 1.35, meaning the summary chunk is usually at rank 1–2. This is very good. [1706][1712]
- **Hybrid**:
  Hit@5 = 0.333 and MRR@5 ≈ 0.081 → hybrid is mostly pulling other chunks to the top, and the summary chunk is often lower or missing within top‑5.

For the **Medications eval question**, semantic retrieval is now clearly the best:

- It reliably finds the patient_overview Medications summary chunk and ranks it very high.
- Lexical and hybrid don’t.

**What this means overall**

With your refined gold sets:

- **Patient Overview**: semantic/hybrid > lexical.
- **Conditions**: semantic > hybrid >> lexical.
- **Medications**: semantic >> hybrid, lexical fails.
- **Oncology timeline** (from earlier): lexical/hybrid were strong because gold was oncology_timeline doc_type and lexical matched those headings well.

So your pipeline is converging to a **task-specific retrieval strategy**:

- Use **semantic** retrieval when the gold chunk lives in **patient_overview summary sections** (Overview, Conditions, Medications).
- Consider **lexical or hybrid** for oncology timeline, where doc_type and headings are very structured and lexical matching is strong.
- Lexical alone is not a good choice for summary-style queries (Overview, Conditions, Medications) when gold is patient_overview.md.

The “worse” numbers (0s instead of 1s) aren’t actually bad—they’re just a more honest reflection of the difficulty of retrieving those specific summary chunks with lexical search. Before you narrowed gold, any conditions.csv chunk was “good enough”; now you’re measuring how often the retriever finds the **exact summary chunk you care about**, and lexical is struggling.

If you’d like, next step would be:

- For one patient, print:
  - The `patient_overview` Medications and Recent Conditions chunks.
  - The top‑5 semantic vs lexical retrieved chunks and their headings.
- That will give you a concrete view of why semantic finds the summary and lexical doesn’t.

### Trying $$k = 5, 7, 10$$ is a good idea. 
It keeps the context window reasonably small (realistic for your RAG pipeline) while letting you see how much improvement you get from slightly larger top‑k sets.

Here’s how to think about those values:

- **k = 5**  
  – Very strict, minimal context. Good for “can we hit the summary with a tiny window?”  
  – You’ve already seen that lexical often fails here for Conditions/Medications.
- **k = 7**  
  – Small increase; sometimes enough to catch a summary chunk that was just outside the top‑5.  
  – Useful to see if “just a bit more context” helps without blowing up tokens.
- **k = 10**  
  – Still manageable for RAG (10 chunks) but gives the retriever more room.  
  – Good to answer: “Are we missing summaries completely, or just ranking them slightly lower?”

You can run your existing eval loop for all three $$k$$ values:

```python
for k in [5, 7, 10]:
    for st in ["lexical", "semantic", "hybrid"]:
        hr, mrr = eval_search_type_df(conditions_gold, st, k=k)
        print(f"[conditions_gold] {st}: Hit@{k} = {hr:.3f}, MRR@{k} = {mrr:.3f}")
```

And similarly for `medications_gold` and `patient_overview_gold`.

What you’re looking for:

- If **lexical Hit@7 or Hit@10** jumps up (e.g., from 0.0 to 0.7), it means summaries are being retrieved, just not in the top‑5.
- If **semantic Hit@5** is already high and doesn’t change much with $$k$$, that reinforces semantic as the better default for those summary-style questions.

So yes: starting with $$k = 5, 7, 10$$ is a sensible, focused way to explore how sensitive your retrieval is to the context window size you’ll realistically use.

In [35]:
# patient_overview_gold
for k in [5, 7, 10]:
    for st in ["lexical", "semantic", "hybrid"]:
        hr, mrr = eval_search_type_df(patient_overview_gold, st, k=k)
        print(f"[patient_overview_gold] {st}: Hit@{k} = {hr:.3f}, MRR@{k} = {mrr:.3f}")

[patient_overview_gold] lexical: Hit@5 = 0.778, MRR@5 = 0.189
[patient_overview_gold] semantic: Hit@5 = 1.000, MRR@5 = 0.289
[patient_overview_gold] hybrid: Hit@5 = 1.000, MRR@5 = 0.313
[patient_overview_gold] lexical: Hit@7 = 1.000, MRR@7 = 0.226
[patient_overview_gold] semantic: Hit@7 = 1.000, MRR@7 = 0.289
[patient_overview_gold] hybrid: Hit@7 = 1.000, MRR@7 = 0.313
[patient_overview_gold] lexical: Hit@10 = 1.000, MRR@10 = 0.226
[patient_overview_gold] semantic: Hit@10 = 1.000, MRR@10 = 0.289
[patient_overview_gold] hybrid: Hit@10 = 1.000, MRR@10 = 0.313


In [74]:
# conditions_gold
for k in [5, 7, 10]:
    for st in ["lexical", "semantic", "hybrid"]:
        hr, mrr = eval_search_type_df(conditions_gold, st, k=k)
        print(f"[conditions_gold] {st}: Hit@{k} = {hr:.3f}, MRR@{k} = {mrr:.3f}")

[conditions_gold] lexical: Hit@5 = 0.111, MRR@5 = 0.037
[conditions_gold] semantic: Hit@5 = 1.000, MRR@5 = 0.422
[conditions_gold] hybrid: Hit@5 = 0.556, MRR@5 = 0.231
[conditions_gold] lexical: Hit@7 = 0.333, MRR@7 = 0.071
[conditions_gold] semantic: Hit@7 = 1.000, MRR@7 = 0.422
[conditions_gold] hybrid: Hit@7 = 0.556, MRR@7 = 0.231
[conditions_gold] lexical: Hit@10 = 0.333, MRR@10 = 0.071
[conditions_gold] semantic: Hit@10 = 1.000, MRR@10 = 0.422
[conditions_gold] hybrid: Hit@10 = 1.000, MRR@10 = 0.280


In [37]:
# medications_gold
for k in [5, 7, 10]:
    for st in ["lexical", "semantic", "hybrid"]:
        hr, mrr = eval_search_type_df(medications_gold, st, k=k)
        print(f"[medications_gold] {st}: Hit@{k} = {hr:.3f}, MRR@{k} = {mrr:.3f}")

[medications_gold] lexical: Hit@5 = 0.000, MRR@5 = 0.000
[medications_gold] semantic: Hit@5 = 1.000, MRR@5 = 0.741
[medications_gold] hybrid: Hit@5 = 0.333, MRR@5 = 0.081
[medications_gold] lexical: Hit@7 = 0.000, MRR@7 = 0.000
[medications_gold] semantic: Hit@7 = 1.000, MRR@7 = 0.741
[medications_gold] hybrid: Hit@7 = 0.889, MRR@7 = 0.166
[medications_gold] lexical: Hit@10 = 0.111, MRR@10 = 0.012
[medications_gold] semantic: Hit@10 = 1.000, MRR@10 = 0.741
[medications_gold] hybrid: Hit@10 = 1.000, MRR@10 = 0.180


In [38]:
# oncology_timeline_gold
for k in [5, 7, 10]:
    for st in ["lexical", "semantic", "hybrid"]:
        hr, mrr = eval_search_type_df(oncology_timeline_gold, st, k=k)
        print(f"[oncology_timeline_gold] {st}: Hit@{k} = {hr:.3f}, MRR@{k} = {mrr:.3f}")

[oncology_timeline_gold] lexical: Hit@5 = 1.000, MRR@5 = 1.000
[oncology_timeline_gold] semantic: Hit@5 = 1.000, MRR@5 = 0.944
[oncology_timeline_gold] hybrid: Hit@5 = 1.000, MRR@5 = 1.000
[oncology_timeline_gold] lexical: Hit@7 = 1.000, MRR@7 = 1.000
[oncology_timeline_gold] semantic: Hit@7 = 1.000, MRR@7 = 0.944
[oncology_timeline_gold] hybrid: Hit@7 = 1.000, MRR@7 = 1.000
[oncology_timeline_gold] lexical: Hit@10 = 1.000, MRR@10 = 1.000
[oncology_timeline_gold] semantic: Hit@10 = 1.000, MRR@10 = 0.944
[oncology_timeline_gold] hybrid: Hit@10 = 1.000, MRR@10 = 1.000


Your reading of these numbers is spot-on, and they give you a clear, data-backed way to choose both **k** and **retrieval mode** per question type.

Here’s the compact interpretation.

Patient overview

- At k=5:
  - Lexical: Hit@5 = 0.778 → misses for 2 patients.
  - Semantic/hybrid: Hit@5 = 1.0, MRR better.
- At k=7:
  - Lexical: Hit@7 = 1.0 → lexical now always finds at least one gold overview chunk.
  - MRR@7 rises slightly (0.226 vs 0.189).
- Semantic/hybrid are unchanged (already perfect at k=5).

This says:

- For **Patient Overview**, k=7 is enough for **lexical** to become reliable, but semantic/hybrid already do fine at k=5.
- In practice, you can keep k=5 for semantic/hybrid; lexical benefits from k=7 if you ever use it for this question.

Conditions BEFORE I FIXED THE TYPE "Recent Conditions" not "Recent Condition"

- At k=5:
  - Lexical: Hit@5 = 0.0 → never finds the summary chunk.
  - Semantic: Hit@5 = 0.778, MRR@5 ≈ 0.289.
- At k=7:
  - Lexical: Hit@7 = 0.111 → rarely finds the chunk (1 of 9 patients).
  - Semantic: Hit@7 = 0.889, MRR@7 ≈ 0.305 → semantic gets stronger.
- At k=10:
  - Lexical: Hit@10 = 0.111 (no real improvement).
  - Semantic: Hit@10 = 0.889, MRR@10 ≈ 0.305.
  - Hybrid: Hit@10 = 0.667, MRR@10 ≈ 0.143 (improves somewhat).

So:

- For **Conditions**, lexical is still weak even at k=10; semantic is clearly best.
- k=7 vs k=10 doesn’t make a huge difference for semantic; it’s already strong at k=5–7.  
- Hybrid improves with larger k, but still lags semantic in both hit and rank.

Medications

- At k=5:
  - Lexical: Hit@5 = 0.0 → never finds the summary chunk.
  - Semantic: Hit@5 = 1.0, MRR@5 ≈ 0.741 → excellent.
  - Hybrid: Hit@5 = 0.333, MRR low.
- At k=7:
  - Lexical: still 0.0.
  - Semantic: unchanged (already perfect).
  - Hybrid: Hit@7 = 0.889, MRR@7 ≈ 0.166 → much better than at k=5.
- At k=10:
  - Lexical: Hit@10 = 0.111 (still weak).
  - Semantic: unchanged (already perfect).
  - Hybrid: Hit@10 = 1.0, MRR@10 ≈ 0.180 → hybrid eventually catches up if k is larger.

So:

- For **Medications**, semantic is excellent even at k=5.
- Hybrid becomes competitive only when you go up to k≈7–10.
- Lexical continues to be poor even at k=10.

Does k=7 “help all”?

### k=7 is a **good compromise**:

- Patient Overview: lexical Hit@7 reaches 1.0; semantic/hybrid unaffected.
- Conditions: semantic Hit@7 improves (0.889 vs 0.778 at k=5); lexical still weak but slightly better.
- Medications: semantic is already perfect; hybrid benefits (Hit@7 0.889 vs 0.333 at k=5), though semantic is still the best choice.

From a practical standpoint:

- For **summary-style questions** (Overview, Conditions, Medications), you’ll often use **semantic** (or hybrid tuned to favor semantic) and k in the **5–7** range:
  - k=5 is fine for Medications and Overview when using semantic/hybrid.
  - k=7 is safer for Conditions if you want to reduce misses.
- For **lexical**, k=7 makes Patient Overview a bit more reliable, but it doesn’t rescue Conditions/Medications enough to make lexical the primary choice for those.

So your intuition is right:

- **k≈7** is a reasonable default for your evaluation runs.
- But the bigger driver of performance is **which retriever you use for which question type**:
  - Overview: semantic or hybrid, k=5–7.
  - Conditions: semantic, k=5–7.
  - Medications: semantic, k=5 (hybrid only if you explicitly want lexical flavor).
  - Oncology timeline: lexical/hybrid (from earlier results), where k=5 may already be enough.

If you’d like, we can now write down a small “retrieval policy” table: for each `question_type`, pick `search_type` and `k` based on these metrics, so you have a clear default for your app and eval code.

### Summary
- For **Patient Overview**, k=7 is enough for **lexical** to become reliable, but semantic/hybrid already do fine at k=5.
- In practice, you can **keep k=5 for semantic/hybrid**; lexical benefits from k=7 if you ever use it for this question.

- For **Conditions**, lexical is still weak even at k=10; **semantic is clearly best**.
- k=7 vs k=10 doesn’t make a huge difference for semantic; it’s **already strong at k=5–7**.  
- Hybrid improves with larger k, but still lags semantic in both hit and rank.

- For **Medications**, **semantic is excellent even at k=5**.
- **Hybrid** becomes competitive only when you go up to **k≈7–10**.
- Lexical continues to be poor even at k=10.

## QUESTION ARCHETYPES

Questions as 4 archetypes:
- Patient overview
- Conditions
- Medications
- Oncology timeline

### 1. QUESTION_TYPES and routing rules

This way, any user question is mapped onto a task type, even if it isn’t exactly one of your four eval questions.

Later, you can replace this with an LLM-powered classifier (see my googledoc notes too)

In [ ]:
# first verions for the LLM judge
QUESTION_TYPES = {
    "patient_overview": {
        "description": "Overview of medical background and current context",
        "doc_types": ["patient_overview"],  # only patient_overview chunks
        "prompt_mode": "summary",
    },
    "conditions": {
        "description": "Diagnosed conditions and statuses",
        "doc_types": ["patient_overview", "conditions"],  # with "conditions" first it listed the most frequent conditions, e.g. violence
        "prompt_mode": "extract_conditions",
    },
    "medications": {
        "description": "Medications the patient is taking or has recently taken",
        "doc_types": ["medications", "patient_overview"],  # meds sources
        "prompt_mode": "extract_medications",
    },
    "oncology_timeline": {
        "description": "Oncology history and major events",
        "doc_types": ["oncology_timeline", "oncology_timeline_events"],
        "prompt_mode": "summarize_oncology_timeline",
    },
    # "current_status": {           # for “current status” questions
    # "description": "Most recent clinical status and events",
    # "doc_types": ["patient_overview", "encounters", "diagnostic_reports"],
    # "prompt_mode": "summarize_current_status",
    # },
}

In [76]:
# question-type classifier

def classify_question_type(question: str) -> str:
    q = question.lower()

    if any(word in q for word in ["overview", "summary", "background", "history"]):
        return "patient_overview"

    if any(word in q for word in ["condition", "diagnosis", "diagnosed"]):
        return "conditions"

    if any(word in q for word in ["medication", "drug", "therapy", "prescription"]):
        return "medications"

    if any(word in q for word in ["oncology", "cancer", "tumor", "chemo", "radiation", "stage"]):
        return "oncology_timeline"

    # fallback
    return "patient_overview"

### 3. Task-specific prompting (different prompt modes)
You can keep INSTRUCTIONS as a base and add per-task “mode” instructions.
Then adjust build_prompt(...) so it injects the right extra instructions -> build_prompt_with_mode


In the descriptions below I first changed:
```
For questions about diagnosed conditions:

- Treat this as an extraction task.
- List ONLY conditions explicitly mentioned in the context.
- For each condition, preserve its status exactly (active or resolved).
- Do NOT add conditions that are not in the context.
```

In [77]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "clinical_synopsis"))

import rag as rag_module

In [78]:
BASE_INSTRUCTIONS = rag_module.INSTRUCTIONS  # your current general instructions

CONDITIONS_EXTRA = """
For questions about diagnosed conditions:

- Treat this as an extraction task based on the context.
- Focus on the patient's main *clinical* conditions (e.g., malignancies, significant comorbidities),
  not social findings or environment reports.
- Use the "Recent Conditions" section from patient_overview.md as the primary source.
- List conditions in chronological order by their documented date (most recent first is acceptable).
- For each condition, preserve its status exactly (active or resolved).
- Do NOT infer conditions that are not mentioned.
- If a date is missing, say "date: not documented" instead of inventing one.
"""

MEDICATIONS_EXTRA = """
For questions about medications:

- List ONLY medications explicitly mentioned in the context.
- Include the medication name and, when available, dose or regimen.
- Do NOT add medications that are not in the context.
"""

ONCOLOGY_TIMELINE_EXTRA = """
For questions about oncology history:

- Summarize the patient’s oncology-related timeline.
- Include major events (diagnosis, staging, treatments, progression or response).
- Ground everything in the context; do NOT invent events.
"""

# For patient_overview, you might keep it more free-form summary but still grounded.
PATIENT_OVERVIEW_EXTRA = """
For overview questions:

- Provide a concise summary of the patient’s medical background and current context.
- Focus on major conditions, treatments, and recent events.
- Base your summary strictly on the context; do NOT add conditions or events not in the context.
"""

PROMPT_MODES = {
    "summary": PATIENT_OVERVIEW_EXTRA,
    "extract_conditions": CONDITIONS_EXTRA,
    "extract_medications": MEDICATIONS_EXTRA,
    "summarize_oncology_timeline": ONCOLOGY_TIMELINE_EXTRA,
}

In [79]:
def build_prompt_with_mode(question, search_results, prompt_mode):
    context = rag_module.build_context(search_results)
    extra = PROMPT_MODES.get(prompt_mode, "")

    prompt = f"""
{extra}

QUESTION: {question}

CONTEXT:
{context}
""".strip()

    return prompt

### 4. Filtered context builder in the notebook

!!!! run these before rag_new

In rag_new, replace:
```
context = rag_module.build_context(search_results)
prompt = build_prompt_with_mode(query, search_results, prompt_mode)
```
with
```
context = build_filtered_context(search_results, question_type=question_type)
prompt = build_prompt_with_mode(query, search_results, prompt_mode)
```


In [ ]:
# def build_filtered_context(search_results, question_type=None):
#     """
#     Wraps rag_module.build_context, with extra filtering for specific question types.
#     For 'conditions', prioritize the 'Recent Conditions' section in patient_overview,
#     and include conditions.csv entries as secondary context.
#     """
#     # if question_type == "conditions":
#     #     filtered = []

#     #     # First: Recent Conditions from patient_overview.md
#     #     for doc in search_results:
#     #         if (
#     #             doc.get("doc_type") == "patient_overview"
#     #             and doc.get("heading") == "Recent Conditions"
#     #         ):
#     #             filtered.append(doc)

#     #     # # Then: conditions.csv entries
#     #     # for doc in search_results:
#     #     #     if doc.get("doc_type") == "conditions":
#     #     #         filtered.append(doc)

#     #     # Fall back: if nothing matched, just use original results
#     #     if filtered:
#     #         return rag_module.build_context(filtered)
#     if question_type == "conditions":
#         filtered = [
#             doc for doc in search_results
#             if doc.get("doc_type") == "patient_overview"
#             and doc.get("heading") == "Recent Conditions"
#         ]

#         if not filtered:
#             # For eval, it's better to fail loudly than silently use the wrong context
#             raise ValueError("No 'Recent Conditions' section found in patient_overview for this patient.")

#             return rag_module.build_context(search_results)

#     # Default behavior for other question types
#     return rag_module.build_context(search_results)

In [89]:
def build_filtered_context(search_results, question_type=None):
    if question_type == "conditions":
        filtered = [
            doc for doc in search_results
            if doc.get("doc_type") == "patient_overview"
            and doc.get("heading") == "Recent Conditions"
        ]

        if filtered:
            return rag_module.build_context(filtered)

        # fallback instead of hard fail
        return rag_module.build_context(search_results)

    return rag_module.build_context(search_results)

### 2. rag_new Task-specific context selection

rag_new:

Reuses all core logic from rag.py (search, semantic_search, hybrid_search, build_context, llm, evaluate_relevance).

Adds only:

QUESTION_TYPES configuration.

prompt_mode routing.

Extra instructions injected into the prompt.

!!!! If you previously did:

```python
from clinical_synopsis.rag import rag
```
then rag in your notebook is the function, not the module, which causes the error you see.

In [90]:
# from pathlib import Path
# import sys

# project_root = Path.cwd().parent
# sys.path.insert(0, str(project_root / "clinical_synopsis"))

# import rag as rag_module



def rag_new(
    query,
    patient_id=None,
    question_type="patient_overview",
    num_results=7,
    model="gpt-5.4-mini",
    search_type="lexical",
):
    """
    End-to-end RAG with task-specific routing:
    - Chooses doc_types and prompt_mode based on question_type.
    - Uses search/semantic/hybrid from clinical_synopsis/rag.py.
    - Uses llm and evaluate_relevance from rag.py.
    """

    from time import time
    t0 = time()

    # Decide doc_types and prompt_mode based on question_type
    q_cfg = QUESTION_TYPES.get(question_type, QUESTION_TYPES["patient_overview"])
    doc_types = q_cfg["doc_types"]
    prompt_mode = q_cfg["prompt_mode"]

    # Retrieval using rag_module functions
    if search_type == "lexical":
        search_results = rag_module.search(
            query=query,
            patient_id=patient_id,
            doc_types=doc_types,
            is_oncology=None,
            num_results=num_results,
        )
    elif search_type == "semantic":
        search_results = rag_module.semantic_search(
            query=query,
            patient_id=patient_id,
            doc_types=doc_types,
            is_oncology=None,
            num_results=num_results,
        )
    elif search_type == "hybrid":
        search_results = rag_module.hybrid_search(
            query=query,
            patient_id=patient_id,
            doc_types=doc_types,
            is_oncology=None,
            num_results=num_results,
        )
    else:
        raise ValueError("search_type must be one of: lexical, semantic, hybrid")

    # Build prompt with mode-specific instructions
    prompt = build_prompt_with_mode(query, search_results, prompt_mode)

    # Call llm from rag_module
    llm_result = rag_module.llm(prompt, model=model)
    answer = llm_result["answer"]
    token_stats = llm_result["token_stats"]
    answer_cost = llm_result["cost"]

    # Build filtered context for relevance/groundedness evaluation
    # context = rag_module.build_context(search_results)
    context = build_filtered_context(search_results, question_type=question_type)
    # print("=== DEBUG CONTEXT START ===")
    # print(context)
    # print("=== DEBUG CONTEXT END ===")

    eval_result = rag_module.evaluate_relevance(
        question=query,
        answer=answer,
        context=context,
        search_type=search_type,
        model=model,
    )
    evaluation = eval_result["evaluation"]
    eval_token_stats = eval_result["token_stats"]
    eval_cost = eval_result["cost"]

    took = time() - t0

    answer_data = {
        "answer": answer,
        "model_used": model,
        "search_type": search_type,
        "question_type": question_type,
        "response_time": took,
        "relevance": evaluation.get("Relevance", "UNKNOWN"),
        "groundedness": evaluation.get("Groundedness", "UNKNOWN"),
        "evaluation_explanation": evaluation.get("Explanation", "Failed to parse evaluation"),
        "search_results": search_results,

        "prompt_tokens": token_stats["input_tokens"],
        "completion_tokens": token_stats["output_tokens"],
        "total_tokens": token_stats["total_tokens"],

        "answer_input_cost_usd": answer_cost["input_cost"],
        "answer_output_cost_usd": answer_cost["output_cost"],
        "answer_total_cost_usd": answer_cost["total_cost"],

        "eval_prompt_tokens": eval_token_stats["input_tokens"],
        "eval_completion_tokens": eval_token_stats["output_tokens"],
        "eval_total_tokens": eval_token_stats["total_tokens"],

        "eval_input_cost_usd": eval_cost["input_cost"],
        "eval_output_cost_usd": eval_cost["output_cost"],
        "eval_total_cost_usd": eval_cost["total_cost"],

        "overall_total_cost_usd": answer_cost["total_cost"] + eval_cost["total_cost"],
    }

    return answer_data

### 5. Test routing in the notebook

In [46]:
patient_ids

['d65197b3-056a-2136-b584-77f43c29da3f',
 'f3739580-797d-ae04-eebf-aeddb2fc2f64',
 '4736727e-63f4-071a-1516-a49310f5a052',
 '29f6beee-162f-0113-7884-72245814693f',
 '41681ed6-efc5-94c0-1bc0-f60b34dbd31b',
 'aee216e6-cbe8-eaf2-3241-4bd1e8a01494',
 'ecc4a7d0-8838-36b4-44ba-676d5a1f7927',
 '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678',
 'f203e11d-5573-1624-69b8-af8436987b3e']

In [82]:
all_gold_df = pd.concat(
    [
        patient_overview_gold,
        conditions_gold,
        medications_gold,
        oncology_timeline_gold,
    ],
    ignore_index=True,
)

all_gold_df

,patient_id,question_type,question_text,gold_chunk_ids
0,29f6beee-162f-0113-7884-72245814693f,patient_overview,Give a concise overview of this patient’s medi...,"[abc678e3f0fdb2313c03b92ff62bf06f08120f7b, c1b..."
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,patient_overview,Give a concise overview of this patient’s medi...,"[a970cc1ae6abca57a8813b4cc2aea2df1d41cde1, 519..."
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,patient_overview,Give a concise overview of this patient’s medi...,"[0590211551c10b6259f5a9062003667e93e8f4e6, 9c7..."
3,4736727e-63f4-071a-1516-a49310f5a052,patient_overview,Give a concise overview of this patient’s medi...,"[63f01e07f0f326ce090ed1d0e1e9cb9ddb88d53b, 49c..."
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,patient_overview,Give a concise overview of this patient’s medi...,"[aff56cb2f4bb43dff398ad5e744d1f5a364b7670, 280..."
5,d65197b3-056a-2136-b584-77f43c29da3f,patient_overview,Give a concise overview of this patient’s medi...,"[abc3ffd40cbb13d210ac910a0f5aaa43b998a587, e84..."
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,patient_overview,Give a concise overview of this patient’s medi...,"[aee14674ce11ab5daa05de7c09f81db1f25365a4, 95b..."
7,f203e11d-5573-1624-69b8-af8436987b3e,patient_overview,Give a concise overview of this patient’s medi...,"[1f9771cf6cdb8353d65819a133ea34b11b3b8e58, 78c..."
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,patient_overview,Give a concise overview of this patient’s medi...,"[a9165d0158523ea3b0c882ba42cd1f37f02f067d, c59..."
9,29f6beee-162f-0113-7884-72245814693f,conditions,What are the patient’s main diagnosed conditions?,"[14326c002878a932e87427ea2a9ba2b6e6ad1404, abc..."


In [83]:
REFERENCE_EVAL_PROMPT_TEMPLATE = """
You are an expert evaluator for a clinical RAG system.

Your task is to evaluate the generated answer against a reference summary
for three criteria:

1. Relevance to the user's question
2. Faithfulness to the reference (no contradictions or hallucinations)
3. Coverage of the key points in the reference that are relevant to the question

Classify relevance as one of:
- "NON_RELEVANT"
- "PARTLY_RELEVANT"
- "RELEVANT"

Classify faithfulness as one of:
- "NOT_FAITHFUL"
- "PARTLY_FAITHFUL"
- "FAITHFUL"

Classify coverage as one of:
- "INSUFFICIENT"
- "PARTIAL"
- "COMPREHENSIVE"

Question: {question}

Generated answer:
{answer}

Reference summary:
{reference}

Return parsable JSON only, without code fences, in exactly this format:

{{
  "Relevance": "NON_RELEVANT" | "PARTLY_RELEVANT" | "RELEVANT",
  "Faithfulness": "NOT_FAITHFUL" | "PARTLY_FAITHFUL" | "FAITHFUL",
  "Coverage": "INSUFFICIENT" | "PARTIAL" | "COMPREHENSIVE",
  "Explanation": "[Provide a brief explanation for your evaluation]"
}}
""".strip()

In [84]:
import json

def evaluate_against_reference(question, answer, reference, model="gpt-5.4-mini"):
    """LLM-as-judge for relevance, faithfulness, and coverage vs a reference summary."""
    prompt = REFERENCE_EVAL_PROMPT_TEMPLATE.format(
        question=question,
        answer=answer,
        reference=reference,
    )

    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "developer",
                "content": [
                    {
                        "type": "input_text",
                        "text": "Return valid JSON only. Do not include markdown or code fences.",
                    }
                ],
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": prompt,
                    }
                ],
            },
        ],
    )

    evaluation_text = response.output_text.strip()

    usage = getattr(response, "usage", None)
    if usage is None:
        token_stats = {
            "input_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
        }
    else:
        input_tokens = getattr(usage, "input_tokens", 0)
        output_tokens = getattr(usage, "output_tokens", 0)
        total_tokens = getattr(usage, "total_tokens", input_tokens + output_tokens)
        token_stats = {
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
        }

    cost_info = calculate_openai_cost(model, token_stats)

    try:
        evaluation = json.loads(evaluation_text)
    except json.JSONDecodeError:
        evaluation = {
            "Relevance": "UNKNOWN",
            "Faithfulness": "UNKNOWN",
            "Coverage": "UNKNOWN",
            "Explanation": "Failed to parse evaluation",
        }

    return {
        "evaluation": evaluation,
        "token_stats": token_stats,
        "cost": cost_info,
        "raw_text": evaluation_text,
    }

In [85]:
import pandas as pd

def build_reference_summary(chunks_df, gold_chunk_ids):
    rows = chunks_df.loc[chunks_df["chunk_id"].isin(gold_chunk_ids)]
    texts = rows["chunk_text"].astype(str).tolist()
    return "\n\n".join(texts)

In [91]:
# from pathlib import Path
# import sys

# project_root = Path.cwd().parent
# sys.path.insert(0, str(project_root / "clinical_synopsis"))

# import rag as rag_module

import math
# import pandas as pd

# expose rag helpers that notebook code expects as globals
calculate_openai_cost = rag_module.calculate_openai_cost
client = rag_module.client

def run_reference_eval(
    gold_df,
    chunks_df,
    search_type="hybrid",
    model="gpt-5.4-mini",
    num_results=5,
):
    """
    Evaluate rag_new() against a reference summary built from gold_chunk_ids.

    gold_df columns:
      - patient_id
      - question_type
      - question_text
      - gold_chunk_ids (list of chunk_ids)
    """
    eval_rows = []

    for _, row in gold_df.iterrows():
        patient_id = row["patient_id"]
        question_type = row["question_type"]
        question = row["question_text"]
        gold_ids = row["gold_chunk_ids"]

        # 1. Build reference summary from gold chunks
        reference = build_reference_summary(chunks_df, gold_ids)

        # 2. Run rag_new to get an answer
        rag_out = rag_new(
            query=question,
            patient_id=patient_id,
            question_type=question_type,  # routing based on archetype
            num_results=num_results,
            model=model,
            search_type=search_type,
        )

        answer = rag_out["answer"]

        # 3. Judge answer vs reference
        judge_out = evaluate_against_reference(
            question=question,
            answer=answer,
            reference=reference,
            model=model,
        )

        evaluation = judge_out["evaluation"]

        eval_rows.append({
            "patient_id": patient_id,
            "question_type": question_type,
            "search_type": search_type,
            "question_text": question,
            "gold_chunk_ids": gold_ids,
            "reference_summary": reference,
            "model_answer": answer,

            # Judge scores
            "judge_relevance": evaluation.get("Relevance", "UNKNOWN"),
            "judge_faithfulness": evaluation.get("Faithfulness", "UNKNOWN"),
            "judge_coverage": evaluation.get("Coverage", "UNKNOWN"),
            "judge_explanation": evaluation.get("Explanation", ""),

            # Answer token / cost (from rag_new / rag_module.llm)
            "prompt_tokens": rag_out.get("prompt_tokens"),
            "completion_tokens": rag_out.get("completion_tokens"),
            "total_tokens": rag_out.get("total_tokens"),

            "answer_input_cost_usd": rag_out.get("answer_input_cost_usd"),
            "answer_output_cost_usd": rag_out.get("answer_output_cost_usd"),
            "answer_total_cost_usd": rag_out.get("answer_total_cost_usd"),

            # Eval token / cost (if you later add them to rag_new; for now these may be None)
            "eval_prompt_tokens": rag_out.get("eval_prompt_tokens"),
            "eval_completion_tokens": rag_out.get("eval_completion_tokens"),
            "eval_total_tokens": rag_out.get("eval_total_tokens"),

            "eval_input_cost_usd": rag_out.get("eval_input_cost_usd"),
            "eval_output_cost_usd": rag_out.get("eval_output_cost_usd"),
            "eval_total_cost_usd": rag_out.get("eval_total_cost_usd"),

            "overall_total_cost_usd": rag_out.get("overall_total_cost_usd"),
        })

    return pd.DataFrame(eval_rows)

In [65]:
# Evaluate patient overview questions with HYBRID search
po_gold = patient_overview_gold[
    patient_overview_gold["question_type"] == "patient_overview"
]

po_ref_eval = run_reference_eval(
    gold_df=po_gold,
    chunks_df=chunks_df,
    search_type="hybrid",
    model="gpt-5.4-mini",
    num_results=5,
)

po_ref_eval

,patient_id,question_type,search_type,question_text,gold_chunk_ids,reference_summary,model_answer,judge_relevance,judge_faithfulness,judge_coverage,...,answer_input_cost_usd,answer_output_cost_usd,answer_total_cost_usd,eval_prompt_tokens,eval_completion_tokens,eval_total_tokens,eval_input_cost_usd,eval_output_cost_usd,eval_total_cost_usd,overall_total_cost_usd
0,29f6beee-162f-0113-7884-72245814693f,patient_overview,hybrid,Give a concise overview of this patient’s medi...,"[abc678e3f0fdb2313c03b92ff62bf06f08120f7b, c1b...",- Total score [DAST-10]; value: 2.0 {score}; d...,This is a female patient born 1980-05-15 with ...,PARTLY_RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001134,0.000652,0.001786,1733,114,1847,0.001300,0.000513,0.001813,0.003599
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,patient_overview,hybrid,Give a concise overview of this patient’s medi...,"[a970cc1ae6abca57a8813b4cc2aea2df1d41cde1, 519...",- Total score [AUDIT-C]; value: 1.0 {score}; d...,This is a female patient born on 1969-05-12. T...,PARTLY_RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001174,0.000639,0.001814,1784,73,1857,0.001338,0.000329,0.001667,0.003480
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,patient_overview,hybrid,Give a concise overview of this patient’s medi...,"[0590211551c10b6259f5a9062003667e93e8f4e6, 9c7...",- Cancer Disease Progression; value: Patient's...,This is an oncology patient overview for a 48-...,PARTLY_RELEVANT,NOT_FAITHFUL,PARTIAL,...,0.001401,0.000981,0.002382,2162,105,2267,0.001621,0.000473,0.002094,0.004476
3,4736727e-63f4-071a-1516-a49310f5a052,patient_overview,hybrid,Give a concise overview of this patient’s medi...,"[63f01e07f0f326ce090ed1d0e1e9cb9ddb88d53b, 49c...",- Cancer Disease Progression; value: Patient's...,This is an oncology patient overview for a fem...,PARTLY_RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001112,0.000652,0.001764,1702,129,1831,0.001276,0.000580,0.001857,0.003621
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,patient_overview,hybrid,Give a concise overview of this patient’s medi...,"[aff56cb2f4bb43dff398ad5e744d1f5a364b7670, 280...",- Total score [DAST-10]; value: 1.0 {score}; d...,This patient is a female born on 1984-06-18. T...,PARTLY_RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001287,0.000765,0.002052,1962,93,2055,0.001472,0.000418,0.001890,0.003942
5,d65197b3-056a-2136-b584-77f43c29da3f,patient_overview,hybrid,Give a concise overview of this patient’s medi...,"[abc3ffd40cbb13d210ac910a0f5aaa43b998a587, e84...",- Cancer Disease Progression; value: Patient's...,This is a young female oncology patient with a...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001060,0.000976,0.002036,1705,88,1793,0.001279,0.000396,0.001675,0.003711
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,patient_overview,hybrid,Give a concise overview of this patient’s medi...,"[aee14674ce11ab5daa05de7c09f81db1f25365a4, 95b...",- Cancer Disease Progression; value: Patient's...,This is a female patient born on 1947-04-19 wi...,RELEVANT,FAITHFUL,COMPREHENSIVE,...,0.001308,0.000558,0.001866,1944,92,2036,0.001458,0.000414,0.001872,0.003738
7,f203e11d-5573-1624-69b8-af8436987b3e,patient_overview,hybrid,Give a concise overview of this patient’s medi...,"[1f9771cf6cdb8353d65819a133ea34b11b3b8e58, 78c...",- Total score [AUDIT-C]; value: 1.0 {score}; d...,This is a female patient born in 1965. The rec...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001530,0.000810,0.002340,2296,65,2361,0.001722,0.000292,0.002014,0.004354
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,patient_overview,hybrid,Give a concise overview of this patient’s medi...,"[a9165d0158523ea3b0c882ba42cd1f37f02f067d, c59...",- Cancer Disease Progression; value: Patient's...,This is an oncology patient with a documented ...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001195,0.000702,0.001897,1826,69,1895,0.001370,0.000311,0.001680,0.003577


f3739580-797d-ae04-eebf-aeddb2fc2f64

answer

This is an oncology patient with a documented history of **acute myeloid leukemia** [from 2019 resolved!]and later **malignant neoplasm of the breast** [active!], both listed in the patient overview’s **Recent Conditions** section. Other recent conditions include **otitis media** (active as of **2022-03-27**) and previously **acute viral pharyngitis** and **neutropenia**, both resolved.

[Encounters are useless]
The **Recent Encounters** section shows multiple hospital encounters at **COOLEY DICKINSON HOSPITAL INC,THE** from **2022-03-02 through 2022-06-21**, including several “encounter for problem” visits and one “encounter for symptom” on **2022-03-27**.

patient_overview.md

## Recent Conditions
- Otitis media; date: 2022-03-27T21:25:37-04:00; status: active
- Malignant neoplasm of breast (disorder); date: 2021-12-31T20:25:37-05:00; status: active
- Acute viral pharyngitis (disorder); date: 2020-07-10T10:25:37-04:00; status: resolved
- Neutropenia (disorder); date: 2019-06-11T21:55:37-04:00; status: resolved
- Acute myeloid leukemia, disease (disorder); date: 2019-06-11T21:25:37-04:00; status: resolved

## Recent Results
- Cancer Disease Progression; value: Patient's condition improved; date: 2022-06-21T04:48:45-04:00
- Treatment status Cancer; value: Treatment changed (situation); date: 2022-06-21T04:48:45-04:00
- Stage group.clinical Cancer; value: Stage 1 (qualifier value); date: 2022-01-02T06:23:56-05:00
- Stage group.clinical Cancer; value: Stage 1A (qualifier value); date: 2022-01-02T06:23:56-05:00
- Progesterone receptor [Interpretation] in Tissue; value: Negative (qualifier value); date: 2022-01-02T06:23:56-05:00
- Estrogen receptor [Interpretation] in Tissue; value: Positive (qualifier value); date: 2022-01-02T06:23:56-05:00
- HER2 [Presence] in Breast cancer specimen by FISH; value: Negative (qualifier value); date: 2022-01-02T06:23:56-05:00
- HER2 [Interpretation] in Tissue; value: Negative (qualifier value); date: 2022-01-02T06:23:56-05:00
- Primary tumor.clinical [Class] Cancer; value: T1 category (finding); date: 2021-12-31T21:31:56-05:00
- Distant metastases.clinical [Class] Cancer; value: M0 category (finding); date: 2021-12-31T21:31:56-05:00
- Size Tumor; date: 2021-12-31T21:31:56-05:00
- Regional lymph nodes.clinical [Class] Cancer; value: N0 category (finding); date: 2021-12-31T21:31:56-05:00

In [92]:
# po_gold = patient_overview_gold[
#     patient_overview_gold["question_type"] == "patient_overview"
# ]

cond_gold = conditions_gold[
    conditions_gold["question_type"] == "conditions"
]

# med_gold = medications_gold[
#     medications_gold["question_type"] == "medications"
# ]

# onc_gold = oncology_timeline_gold[
#     oncology_timeline_gold["question_type"] == "oncology_timeline"
# ]
 
# Evaluate condition questions with SEMANTIC search


cond_ref_eval_semantic = run_reference_eval(
    gold_df=cond_gold,
    chunks_df=chunks_df,
    search_type="semantic",
    model="gpt-5.4-mini",
    num_results=5,
)

cond_ref_eval_semantic



,patient_id,question_type,search_type,question_text,gold_chunk_ids,reference_summary,model_answer,judge_relevance,judge_faithfulness,judge_coverage,...,answer_input_cost_usd,answer_output_cost_usd,answer_total_cost_usd,eval_prompt_tokens,eval_completion_tokens,eval_total_tokens,eval_input_cost_usd,eval_output_cost_usd,eval_total_cost_usd,overall_total_cost_usd
0,29f6beee-162f-0113-7884-72245814693f,conditions,semantic,What are the patient’s main diagnosed conditions?,"[14326c002878a932e87427ea2a9ba2b6e6ad1404, abc...",- Not in labor force (finding); date: 2022-06-...,Main diagnosed conditions from **patient_overv...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001729,0.001377,0.003106,943,120,1063,0.000707,0.000540,0.001247,0.004354
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,conditions,semantic,What are the patient’s main diagnosed conditions?,"[772ce44ecf1664ce3cd0f321962e31b1afeaf658, a97...",- Victim of intimate partner abuse (finding); ...,The patient’s main diagnosed conditions listed...,RELEVANT,FAITHFUL,COMPREHENSIVE,...,0.001460,0.001705,0.003165,1029,59,1088,0.000772,0.000266,0.001037,0.004202
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,conditions,semantic,What are the patient’s main diagnosed conditions?,"[8123905ce9ef5c2a100a0400f86029469183dbb2, 059...",- Full-time employment (finding); date: 2022-0...,"The patient’s main diagnosed conditions, from ...",RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001706,0.000837,0.002543,829,85,914,0.000622,0.000383,0.001004,0.003547
3,4736727e-63f4-071a-1516-a49310f5a052,conditions,semantic,What are the patient’s main diagnosed conditions?,"[7ac2bda6270220cb79ace629a38ef8b5a13ddf31, 63f...",- Facial laceration; date: 2022-01-30T15:16:05...,I don’t know.,PARTLY_RELEVANT,FAITHFUL,INSUFFICIENT,...,0.001473,0.000068,0.001540,1972,84,2056,0.001479,0.000378,0.001857,0.003397
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,conditions,semantic,What are the patient’s main diagnosed conditions?,"[e733276d7db1b542f556b25839e3602be1339cbc, aff...",- Sprain of wrist; date: 2022-05-31T18:08:50-0...,I don’t know.,RELEVANT,FAITHFUL,INSUFFICIENT,...,0.001503,0.000068,0.001571,2012,78,2090,0.001509,0.000351,0.001860,0.003430
5,d65197b3-056a-2136-b584-77f43c29da3f,conditions,semantic,What are the patient’s main diagnosed conditions?,"[45b0f62c2348b66d770620e4e5acfe630cf41f96, abc...",- Otitis media; date: 2021-06-16T04:22:18-04:0...,The patient’s main diagnosed conditions docume...,RELEVANT,FAITHFUL,COMPREHENSIVE,...,0.001631,0.000450,0.002080,2273,78,2351,0.001705,0.000351,0.002056,0.004136
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,conditions,semantic,What are the patient’s main diagnosed conditions?,"[05d547f22443d60a0d31fa1776bed869da7459cd, aee...",- Malignant neoplasm of breast (disorder); dat...,The patient’s main diagnosed clinical conditio...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001479,0.001012,0.002491,2190,82,2272,0.001643,0.000369,0.002012,0.004503
7,f203e11d-5573-1624-69b8-af8436987b3e,conditions,semantic,What are the patient’s main diagnosed conditions?,"[f73c3623d5df217075882e25106a9d6388dde40e, 1f9...",- Concussion with loss of consciousness; date:...,The patient’s main diagnosed conditions in the...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001414,0.001422,0.002836,961,70,1031,0.000721,0.000315,0.001036,0.003872
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,conditions,semantic,What are the patient’s main diagnosed conditions?,"[4c17e4784d257f85e9fd1ddd74054e5dc3adae9f, a91...",- Otitis media; date: 2022-03-27T21:25:37-04:0...,The patient’s main diagnosed conditions docume...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001492,0.000630,0.002122,2123,118,2241,0.001592,0.000531,0.002123,0.004246


In [88]:
# 1) Show active function source (to verify which version is loaded)
import inspect
print(inspect.getsource(build_filtered_context))

# 2) Inspect what is actually retrieved for one failing case
test_row = cond_gold.iloc[0]
out = rag_new(
    query=test_row["question_text"],
    patient_id=test_row["patient_id"],
    question_type="conditions",
    search_type="semantic",
    num_results=5,
    model="gpt-5.4-mini",
)
[(d.get("doc_type"), repr(d.get("heading"))) for d in out["search_results"]]

def build_filtered_context(search_results, question_type=None):
    """
    Wraps rag_module.build_context, with extra filtering for specific question types.
    For 'conditions', prioritize the 'Recent Conditions' section in patient_overview,
    and include conditions.csv entries as secondary context.
    """
    # if question_type == "conditions":
    #     filtered = []

    #     # First: Recent Conditions from patient_overview.md
    #     for doc in search_results:
    #         if (
    #             doc.get("doc_type") == "patient_overview"
    #             and doc.get("heading") == "Recent Conditions"
    #         ):
    #             filtered.append(doc)

    #     # # Then: conditions.csv entries
    #     # for doc in search_results:
    #     #     if doc.get("doc_type") == "conditions":
    #     #         filtered.append(doc)

    #     # Fall back: if nothing matched, just use original results
    #     if filtered:
    #         return rag_module.build_context(filte

[('patient_overview', "'Record Snapshot'"),
 ('patient_overview', "'Recent Results'"),
 ('patient_overview', "'Recent Encounters'"),
 ('patient_overview', "'Recent Conditions'"),
 ('patient_overview', "'Diagnostic Reports'")]

In [67]:
# po_gold = patient_overview_gold[
#     patient_overview_gold["question_type"] == "patient_overview"
# ]

# cond_gold = conditions_gold[
#     conditions_gold["question_type"] == "conditions"
# ]

med_gold = medications_gold[
    medications_gold["question_type"] == "medications"
]

# onc_gold = oncology_timeline_gold[
#     oncology_timeline_gold["question_type"] == "oncology_timeline"
# ]
 
# Evaluate medication questions with SEMANTIC search


med_ref_eval_semantic = run_reference_eval(
    gold_df=med_gold,
    chunks_df=chunks_df,
    search_type="semantic",
    model="gpt-5.4-mini",
    num_results=5,
)

med_ref_eval_semantic



,patient_id,question_type,search_type,question_text,gold_chunk_ids,reference_summary,model_answer,judge_relevance,judge_faithfulness,judge_coverage,...,answer_input_cost_usd,answer_output_cost_usd,answer_total_cost_usd,eval_prompt_tokens,eval_completion_tokens,eval_total_tokens,eval_input_cost_usd,eval_output_cost_usd,eval_total_cost_usd,overall_total_cost_usd
0,29f6beee-162f-0113-7884-72245814693f,medications,semantic,What medications is the patient taking or has ...,[313381cf2c6383b0d66557ee95469ebb92cf7cc1],- Camila 28 Day Pack; date: 2022-01-21T13:18:3...,The patient is taking or has recently taken:\n...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001306,0.001404,0.002710,2141,77,2218,0.001606,0.000347,0.001952,0.004663
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,medications,semantic,What medications is the patient taking or has ...,[3f1c94dc52eb9d6748171ca99d8d35ee5b0cc2cd],- amLODIPine 2.5 MG Oral Tablet; date: 2022-01...,The medications explicitly mentioned in the co...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001373,0.000454,0.001827,2017,83,2100,0.001513,0.000373,0.001886,0.003713
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,medications,semantic,What medications is the patient taking or has ...,[392065e296a5586b78eb9e2a43d08df5220cb956],- Camila 28 Day Pack; date: 2021-12-22T21:48:2...,The medications explicitly mentioned in the co...,RELEVANT,PARTLY_FAITHFUL,COMPREHENSIVE,...,0.001546,0.000986,0.002531,2366,71,2437,0.001775,0.000320,0.002094,0.004625
3,4736727e-63f4-071a-1516-a49310f5a052,medications,semantic,What medications is the patient taking or has ...,[7d4fb354f104d3cc11273d15182d3f548ca4031b],- Ibuprofen 100 MG Oral Tablet; date: 2022-01-...,The patient has recently taken or been taking ...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001270,0.000432,0.001702,1875,63,1938,0.001406,0.000284,0.001690,0.003391
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,medications,semantic,What medications is the patient taking or has ...,[3b6a76df9116342cca819342b5a77a629c04d636],- Acetaminophen 325 MG Oral Tablet; date: 2022...,The patient has recently taken or is taking th...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001450,0.001597,0.003047,2374,71,2445,0.001780,0.000320,0.002100,0.005147
5,d65197b3-056a-2136-b584-77f43c29da3f,medications,semantic,What medications is the patient taking or has ...,[24c7158a723f3ec52139411b806ca04fe110024e],- ribociclib 200 MG Oral Tablet; date: 2022-05...,The patient is taking or has recently taken th...,RELEVANT,FAITHFUL,PARTIAL,...,0.001410,0.000481,0.001891,2073,98,2171,0.001555,0.000441,0.001996,0.003887
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,medications,semantic,What medications is the patient taking or has ...,[e3fbadc0fd8c550e6501a326f6222e5c97538c8d],- 5 ML hyaluronidase-oysk 2000 UNT/ML / trastu...,The patient is taking or has recently taken th...,RELEVANT,FAITHFUL,COMPREHENSIVE,...,0.001370,0.001120,0.002490,2162,72,2234,0.001621,0.000324,0.001945,0.004436
7,f203e11d-5573-1624-69b8-af8436987b3e,medications,semantic,What medications is the patient taking or has ...,[e0893f6f1f540051fea4cc26cef7068cb59b41c6],- 5 ML fulvestrant 50 MG/ML Prefilled Syringe;...,Medications explicitly mentioned in the record...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001327,0.001377,0.002704,2161,65,2226,0.001621,0.000292,0.001913,0.004617
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,medications,semantic,What medications is the patient taking or has ...,[be283721d4470a4515d90787d7ce60f5a92db51e],- 5 ML fulvestrant 50 MG/ML Prefilled Syringe;...,The patient has recently taken or is taking th...,RELEVANT,PARTLY_FAITHFUL,PARTIAL,...,0.001310,0.001368,0.002678,2137,83,2220,0.001603,0.000373,0.001976,0.004655


Dropping the LLM judge for now and focusing on **making the answers structurally correct and clinically sensible** is a good move. What you’re seeing—“judge likes most outputs, but I see gross inaccuracies/omissions”—is common when the judge prompt isn’t perfectly aligned with your gold truth and when the context itself is noisy.

You’ve already identified two concrete needs:

1. **Consistent, template-like answer formats per question type.**
2. **Better control over what document types and sections appear in the Patient Overview answer** (e.g., Encounters may be noise for that archetype).

Let’s tackle both.

***

# 1. Template answer formats per question type

Right now, your model is free-form: it decides how to structure the answer. For eval and for your app, you can do better by giving it a **strong answer template** for each archetype.

For example:

### Patient overview template

Structure:

- A short summary paragraph.
- Bullet points grouping by theme: Conditions, Medications, Oncology timeline.

Prompt snippet for `prompt_mode="summary"`:

```text
Format your answer in this structure:

1. **Summary**  
   Provide 2–3 sentences summarizing the patient's overall medical background and current context.

2. **Conditions**  
   List the main diagnosed conditions with their status (active/resolved) and approximate dates, based on the "Recent Conditions" or "Recent Results" sections in the context.

3. **Medications**  
   Summarize current or recent medications from the Medications section (if present).

4. **Oncology timeline**  
   Briefly describe the key oncology-related events (diagnosis, treatments, notable changes) in chronological order.

Do not add sections that are not present in the context.
```

Then your `build_prompt_with_mode` injects this template for `prompt_mode="summary"`.

### Conditions template

Structure:

- A simple list, one line per condition.

Prompt snippet for `prompt_mode="extract_conditions"`:

```text
Format your answer as:

- One bullet per condition.
- Each bullet: **Condition name** — **status**; date: YYYY-MM-DD or "not documented".

Use ONLY conditions from the "Recent Conditions" or "Recent Results" parts of the patient overview section in the context. Do not include conditions that only appear in encounters or other sections.
```

### Medications template

Similar:

```text
Format your answer as:

- One bullet per medication.
- Each bullet: **Medication name** — dose/regimen (if available); status: current or recent; date: YYYY-MM-DD or "not documented".

Use the Medications section in patient_overview.md as the primary source.
```

### Oncology timeline template

```text
Format your answer as a chronological list of key oncology events:

- Each item: date — event (e.g., diagnosis, staging, treatment start/change, progression/response).

Use oncology_timeline and oncology_timeline_events chunks only.
```

By enforcing templates, you:

- Make answers easier to scan and compare.
- Reduce free-form wandering (like mixing in Encounters into the overview unless truly relevant).
- Make human review much easier.

***

## 2. Controlling Encounters in Patient Overview

Your example shows:

> “The Recent Encounters section shows multiple hospital encounters…”  
> [Encounters are useless]

This suggests two things:

- Your **context** for Patient Overview currently includes the `Recent Encounters` chunk from `patient_overview` or from encounters doc_type.
- Your **summary prompt** doesn’t tell the model how to treat encounters (e.g., only mention them if they add clinical context).

If you want Patient Overview answers to **exclude Encounters by default**, you can:

### A. Adjust context selection for `question_type="patient_overview"`

In your `build_filtered_context` or in `rag_new`, for `patient_overview`:

- Only include patient_overview chunks with headings you care about (Conditions, Medications, Oncology Results).
- Exclude Encounters and Provenance from the context used for the summary.

Example:

```python
def build_filtered_context(search_results, question_type=None):
    if question_type == "patient_overview":
        allowed_headings = ["Recent Condition", "Recent Results", "Procedures", "Medications"]
        filtered = [
            doc for doc in search_results
            if doc.get("doc_type") == "patient_overview"
            and doc.get("heading") in allowed_headings
        ]
        if filtered:
            return rag_module.build_context(filtered)
        return rag_module.build_context(search_results)

    # existing logic for conditions/medications/oncology_timeline
    return rag_module.build_context(search_results)
```

This prevents Encounters from appearing in the **Patient Overview** context unless you explicitly add them later.

### B. Adjust the summary prompt to down-weight Encounters

If you decide you sometimes want Encounters (e.g., for “current status” questions), you can add guidance:

```text
In the Summary section, focus on diagnoses, treatments, and major events.

Do NOT list every encounter or visit; only mention encounter details if they are crucial to understanding the patient’s current situation.
```

Given your comment (“Encounters are useless” for the archetype), for now I’d exclude Encounters from the overview context entirely.

***

## 3. How to proceed without relying on the judge

Your plan to:

> “stop looking at the LLM judge from now on, but focus on improving the output as I check it against the records”

is sensible. A practical workflow now is:

1. For each question archetype (Overview, Conditions, Medications, Oncology timeline):
   - Tighten gold truth (you’ve done this).
   - Tighten context selection (summary-first, doc_type/heading filters).
   - Add a strong answer template (format + constraints).
2. For a small set of patients (e.g., your 9 eval patients):
   - Generate answers with `rag_new`.
   - Compare them manually against patient_overview.md and CSVs.
   - Iterate on templates and context filters until outputs match your expectations.

You can still keep the judge around as a **secondary signal** later, but right now human review is the fastest way to see where the LLM is still hallucinating or omitting important facts.

If you’d like, the next concrete step could be:

- Pick one patient.
- Show me the current patient_overview context for the overview archetype and your current summary prompt.
- I can then propose a more constrained template and filter that’s tailored to the actual headings and chunk_text you’re seeing.

# LATER Prioritize patient_overview in RAG context and prompt
while keeping conditions.csv and medications.csv for gold

```python
def build_filtered_context(search_results, question_type=None):
    if question_type == "medications":
        primary = [
            doc for doc in search_results
            if doc.get("doc_type") == "patient_overview"
            and doc.get("heading") == "Medications"
        ]
        secondary = [
            doc for doc in search_results
            if doc.get("doc_type") == "medications"
        ]

        # If we found summary meds, use them first and append csv details after
        if primary:
            ordered = primary + secondary
            return rag_module.build_context(ordered)

        # Fallback: no summary meds found, use whatever we have
        return rag_module.build_context(search_results)

    # existing conditions logic, etc.
    return rag_module.build_context(search_results)
```

In [ ]:
# def build_filtered_context(search_results, question_type=None):
#     if question_type == "medications":
#         primary = [
#             doc for doc in search_results
#             if doc.get("doc_type") == "patient_overview"
#             and doc.get("heading") == "Medications"
#         ]
#         secondary = [
#             doc for doc in search_results
#             if doc.get("doc_type") == "medications"
#         ]

#         # If we found summary meds, use them first and append csv details after
#         if primary:
#             ordered = primary + secondary
#             return rag_module.build_context(ordered)

#         # Fallback: no summary meds found, use whatever we have
#         return rag_module.build_context(search_results)

#     # existing conditions logic, etc.
#     return rag_module.build_context(search_results)